# ExpresSo Split-Horizon Forecast: Short-Horizon AutoGluon + Long-Horizon Candidate

This notebook keeps the structure of `Expresso-AutoGluon-WeightedEnsemble-LeakageSafe-Horizon.ipynb`, but trains two separate horizon-specific predictors:

- `1d` + `7d`: short-horizon AutoGluon weighted ensemble.
- `1m`: long-horizon AutoGluon weighted ensemble, configured as the production-safe choice while TimesFM remains a candidate benchmark.

Why not make TimesFM the default `1m` model here? TimesFM is attractive for long-horizon zero-shot forecasting, but this competition pipeline relies heavily on known covariates such as promotions, holidays, stockout as-of signals, store metadata, weather climatology, and event relevance. AutoGluon consumes these covariates directly and preserves the existing horizon-safe cutoff logic. TimesFM can still be tested later as a separate benchmark or blend, especially if the environment has the current `timesfm` package, Hugging Face access, and XReg dependencies available.


## Public Data References Kept In This Notebook

These are the sources this notebook is designed to use or encode as known-future covariates.

| Feature family | Primary source | How it is used |
|---|---|---|
| Bangkok daily temperature, humidity, precipitation | Open-Meteo Historical Weather API: https://open-meteo.com/en/docs/historical-weather-api | Daily `temperature_2m_*`, `relative_humidity_2m_mean`, `precipitation_sum`, `precipitation_hours` for Bangkok coordinates. |
| Observed climatology / average mean surface air temperature | World Bank Climate Change Knowledge Portal, Thailand climatology: https://climateknowledgeportal.worldbank.org/country/thailand/climate-data-historical | Optional climatology fallback / monthly context if actual weather is too leaky. |
| PTT/OR oil prices | PTTOR SOAP oil price service: https://orapiweb.pttor.com/oilservice/OilPrice.asmx?op=GetOilPrice | Daily PTT fuel prices where the service responds; forward-filled across dates. |
| Competitor oil prices | Bangchak historical retail oil prices: https://www.bangchak.co.th/en/oilprice/historical | Competitor fuel price proxy and PTT-vs-competitor deltas. |
| Thai public holidays | Bank of Thailand financial institution holidays: https://www.bot.or.th/en/financial-institutions-holiday.html and competition `DATE_DIM` | Public holiday and long-weekend features. |
| Civil servant / pension payment cadence | Comptroller General / public payroll calendars, also summarized in Thai financial press such as https://en.moneyandbanking.co.th/2024/141179/ | Salary-day flags and distance-to-payday features. |
| Bangkok events | Visit Bangkok festival calendar, event official pages, and existing `analysis/coffee_hackathon_special_events.py` | Known citywide demand shocks such as Awakening Bangkok, Loy Krathong, Motor Expo, Red Cross Fair, Christmas, New Year countdown. |
| University commencement/graduation | University announcement pages when available | Optional date shocks for university-format stores; included as extensible curated events. |
| Inthanin Coffee competitor context | Bangchak / Inthanin public reports and business updates | Branch-count and station-channel pressure proxy for Bangchak-linked coffee competition. |
| PunThai Coffee competitor context | PTG Energy one report / investor materials and business updates | Branch-count, gas-station/outside-station mix, and PTG-linked coffee competition proxy. |
| PTG / PT Max Station oil-price context | PTG Energy / PT Max Station public oil-price pages when available | Optional future extension; notebook currently uses PTT and Bangchak historical prices because they are cacheable for 2023-2024. |

A separate human-readable copy is written to `outputs/autogluon_external_feature_references.md` by one of the cells below.

## 0. Install / Imports

Use the existing `.venv-autogluon` kernel if available. If running in Colab, uncomment the install line.

In [3]:
# !pip -q install autogluon.timeseries "torch<2.10" torchvision torchaudio lxml beautifulsoup4


In [4]:
# Colab-only install, if needed:
# !pip -q install autogluon.timeseries "torch<2.10" torchvision torchaudio lxml beautifulsoup4

import html
import json
import math
import os
import re
import time
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from pathlib import Path

os.environ.setdefault('LOKY_MAX_CPU_COUNT', '4')
os.environ.setdefault('MPLCONFIGDIR', '/private/tmp/matplotlib-expresso')

import numpy as np
import pandas as pd

from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)
print('Libraries loaded')

Libraries loaded


## 1. Configuration

Set `RUN_TRAINING = True` when you are ready to fit AutoGluon. The defaults are practical for local iteration; increase `TIME_LIMIT_SECONDS` and change `PRESETS` to `best_quality` for a final run.

In [24]:
DATA_DIR_NAME = 'super-ai-engineer-season-6-coffee-chain-hackathon'
ROOT_CANDIDATES = [
    Path.cwd(),
    Path('/Users/moi/PROJECT/Expresso'),
    Path.home() / 'PROJECT' / 'Expresso',
]
ROOT_CANDIDATES += list(Path.cwd().parents)
ROOT = next((candidate for candidate in dict.fromkeys(ROOT_CANDIDATES) if (candidate / DATA_DIR_NAME).exists()), Path.cwd())
DATA_DIR = ROOT / DATA_DIR_NAME
if not DATA_DIR.exists():
    raise FileNotFoundError(f'Could not locate {DATA_DIR_NAME}; checked ROOT candidates: {ROOT_CANDIDATES[:4]} ...')
TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR = DATA_DIR / 'test'
OUTPUT_DIR = ROOT / 'outputs' / 'autogluon_weighted_leakage_safe_horizon'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FORECAST_START = pd.Timestamp('2024-11-01')
FORECAST_END = pd.Timestamp('2024-12-31')
TRAIN_END = pd.Timestamp('2024-10-31')
PREDICTION_LENGTH = (FORECAST_END - FORECAST_START).days + 1
HORIZON_TO_DAYS = {'1d': 1, '7d': 7, '1m': 30}

CAT_ORDER = [
    'Coffee',
    'Tea',
    'Bakery',
    'Savory Bakery',
    'Chocolate & Milk',
    'Juice & Smoothie',
    'Merchandise',
]
DOW_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Strict leakage-safe mode: use climatology and do not train on future oil prices.
WEATHER_MODE = 'climatology'
FETCH_PTT_OIL = False
FETCH_BANGCHAK_OIL = False
RUN_TRAINING = True
RUN_FEATURE_IMPORTANCE = True

TIME_LIMIT_SECONDS = 7200
FEATURE_IMPORTANCE_TIME_LIMIT_SECONDS = 1800
PRESETS = "high_quality"

SHORT_HORIZONS = ['1d', '7d']
LONG_HORIZONS = ['1m']
HORIZON_MODEL_ROUTE = {h: 'short' for h in SHORT_HORIZONS} | {h: 'long' for h in LONG_HORIZONS}

# Short horizon: keep the fast covariate-heavy ensemble. Direct/Recursive tabular models
# are strong when recent calendar, promo, stockout, and event covariates matter.
SHORT_HORIZON_HYPERPARAMETERS = {
    'DirectTabular': {},
    'RecursiveTabular': {},
    'Chronos': {
        'ag_args': {'name_suffix': 'ShortWithRegressor'},
        'model_path': 'bolt_small',
        'target_scaler': 'standard',
        'covariate_regressor': {'model_name': 'CAT', 'model_hyperparameters': {'iterations': 1000}},
    },
}

# Long horizon: still use AutoGluon as the production-safe default because it consumes
# the engineered future covariates and static store/category features without changing
# the leakage-safe prediction contract. Chronos gets a slightly stronger regressor;
# SeasonalNaive is included as a low-variance anchor for the weighted ensemble.
LONG_HORIZON_HYPERPARAMETERS = {
    'DirectTabular': {},
    'RecursiveTabular': {},
    'SeasonalNaive': {},
    'Chronos': {
        'ag_args': {'name_suffix': 'LongWithRegressor'},
        'model_path': 'bolt_small',
        'target_scaler': 'standard',
        'covariate_regressor': {'model_name': 'CAT', 'model_hyperparameters': {'iterations': 1500}},
    },
}

# Keep TimesFM explicit as a candidate, not the default production path.
# Set this to True only after installing/testing timesfm + XReg dependencies in the active kernel.
RUN_EXPERIMENTAL_TIMESFM_1M = False

SHORT_MODEL_PATH = OUTPUT_DIR / 'model_split_horizon_short_1d_7d_autogluon'
LONG_MODEL_PATH = OUTPUT_DIR / 'model_split_horizon_long_1m_autogluon'

# Product-subcategory cluster features from user-marked 11 groups on `04_combined_pca_clusters`.
# These are static category-composition features for the final forecast cutoff.
USE_SUBCATEGORY_CLUSTER_FEATURES = True
SUBCATEGORY_CLUSTER_PATH = ROOT / 'outputs' / 'eda' / 'subcategory_hypothesis' / 'product_group_manual_pca11_clusters.csv'
SUBCATEGORY_CLUSTER_OUTPUT_PATH = OUTPUT_DIR / 'category_subcategory_cluster_features.csv'
SERVE_TYPE_WEATHER_OUTPUT_PATH = OUTPUT_DIR / 'category_serve_type_weather_features.csv'

STOCKOUT_ASOF_FEATURES = [
    'hist_stockout_rate', 'hist_stockout_sku_share', 'hist_mean_closing_stock',
    'stockout_rate_7d', 'stockout_rate_28d',
    'stockout_sku_share_7d', 'stockout_sku_share_28d',
    'closing_stock_mean_7d', 'closing_stock_mean_28d',
    'days_since_stockout',
]

print(f'Prediction length: {PREDICTION_LENGTH} days')
print(f'Output dir: {OUTPUT_DIR}')


Prediction length: 61 days
Output dir: /Users/moi/PROJECT/Expresso/outputs/autogluon_weighted_leakage_safe_horizon


## 2. Read Competition Data And Build Canonical Target

The target is transaction-derived, not inventory-derived. Inventory `units_sold` is not an exact target and should be treated as an explanatory/censoring signal only.

In [6]:
def read_train(name, **kwargs):
    return pd.read_csv(TRAIN_DIR / f'{name}.csv', **kwargs)


def read_test(name, **kwargs):
    return pd.read_csv(TEST_DIR / f'{name}.csv', **kwargs)


product = read_train('PRODUCT')
store = read_train('STORE', parse_dates=['opened_date'])
date_dim = read_train('DATE_DIM', parse_dates=['date'])
order = read_train('ORDER', parse_dates=['date'])
transaction = read_train('TRANSACTION')
inventory = read_train('INVENTORY', parse_dates=['date'])
promotion = read_train('PROMOTION', parse_dates=['start_date', 'end_date'])
local_event = read_train('LOCAL_EVENT', parse_dates=['date'])
sample = pd.read_csv(DATA_DIR / 'sample_submission_with_id.csv')

print({
    'product': product.shape,
    'store': store.shape,
    'date_dim': date_dim.shape,
    'order': order.shape,
    'transaction': transaction.shape,
    'inventory': inventory.shape,
    'promotion': promotion.shape,
    'local_event': local_event.shape,
    'sample': sample.shape,
})

{'product': (60, 7), 'store': (20, 8), 'date_dim': (731, 12), 'order': (1376133, 7), 'transaction': (2858050, 6), 'inventory': (804000, 9), 'promotion': (31314, 10), 'local_event': (1440, 5), 'sample': (25620, 2)}


In [7]:
def build_target_panel(order_df, transaction_df, product_df, store_df):
    txn_order = transaction_df.merge(
        order_df[['order_id', 'store_id', 'date', 'hour', 'customer_id', 'is_member']],
        on='order_id',
        how='left',
    )
    txn_prod = txn_order.merge(
        product_df[['product_id', 'category', 'base_price']],
        on='product_id',
        how='left',
    )
    raw = (
        txn_prod.groupby(['store_id', 'category', 'date'], observed=True)['units_sold']
        .sum()
        .rename('units_sold')
        .reset_index()
    )
    all_dates = pd.date_range('2023-01-01', TRAIN_END, freq='D')
    full_index = pd.MultiIndex.from_product(
        [sorted(store_df['store_id'].unique()), CAT_ORDER, all_dates],
        names=['store_id', 'category', 'date'],
    )
    panel = raw.set_index(['store_id', 'category', 'date']).reindex(full_index, fill_value=0).reset_index()
    panel = panel.merge(store_df[['store_id', 'opened_date']], on='store_id', how='left')
    panel['effective_opened_date'] = panel['opened_date'].clip(lower=pd.Timestamp('2023-01-01'))
    panel['is_active_train_day'] = panel['date'].ge(panel['effective_opened_date'])
    panel = panel[panel['is_active_train_day']].copy()
    panel['item_id'] = panel['store_id'].astype(str) + '_' + panel['category'].str.replace(r'[^A-Za-z0-9]+', '-', regex=True).str.strip('-')
    panel = panel.rename(columns={'date': 'timestamp'})
    return panel, txn_prod


target_panel, txn_prod = build_target_panel(order, transaction, product, store)
print(target_panel.shape)
target_panel.head()

(92694, 8)


,store_id,category,timestamp,units_sold,opened_date,effective_opened_date,is_active_train_day,item_id
0,1,Coffee,2023-01-01,100,2019-12-21,2023-01-01,True,1_Coffee
1,1,Coffee,2023-01-02,140,2019-12-21,2023-01-01,True,1_Coffee
2,1,Coffee,2023-01-03,109,2019-12-21,2023-01-01,True,1_Coffee
3,1,Coffee,2023-01-04,116,2019-12-21,2023-01-01,True,1_Coffee
4,1,Coffee,2023-01-05,136,2019-12-21,2023-01-01,True,1_Coffee


In [8]:
def build_subcategory_cluster_features(product_df, cluster_path, output_path=None):
    """Aggregate product-level combined stock+sales clusters to category-level static features.

    Forecast grain is store_id x category x horizon, while the PCA clusters are product-group level.
    We therefore expose each category's composition across the discovered product clusters.
    """
    base_cols = ['category']
    if not USE_SUBCATEGORY_CLUSTER_FEATURES:
        return pd.DataFrame({'category': CAT_ORDER})

    if not Path(cluster_path).exists():
        print(f'Subcategory cluster file not found: {cluster_path}; skipping subcategory cluster features')
        return pd.DataFrame({'category': CAT_ORDER})

    cluster_df = pd.read_csv(cluster_path)
    cluster_col = 'manual_pca11_cluster' if 'manual_pca11_cluster' in cluster_df.columns else 'combined_cluster'
    required = {'product_group', cluster_col}
    missing = required - set(cluster_df.columns)
    if missing:
        raise ValueError(f'Subcategory cluster file missing columns: {sorted(missing)}')

    prod = product_df.copy()
    prod['product_group'] = prod['product_name'].astype(str)
    cluster_lookup = cluster_df[['product_group', cluster_col]].rename(columns={cluster_col: 'combined_cluster'})
    prod = prod.merge(
        cluster_lookup,
        on='product_group',
        how='left',
    )
    print(f'Using subcategory cluster column: {cluster_col}')
    if prod['combined_cluster'].isna().any():
        missing_products = sorted(prod.loc[prod['combined_cluster'].isna(), 'product_group'].unique())
        raise ValueError(f'Missing subcategory clusters for product groups: {missing_products}')

    prod['combined_cluster'] = prod['combined_cluster'].astype(int)
    cluster_counts = (
        prod.groupby(['category', 'combined_cluster'], observed=True)['product_id']
        .nunique()
        .rename('product_count')
        .reset_index()
    )
    category_totals = cluster_counts.groupby('category', observed=True)['product_count'].sum().rename('category_product_count').reset_index()
    cluster_counts = cluster_counts.merge(category_totals, on='category', how='left')
    cluster_counts['product_share'] = cluster_counts['product_count'] / cluster_counts['category_product_count'].replace(0, np.nan)

    shares = cluster_counts.pivot(index='category', columns='combined_cluster', values='product_share').fillna(0)
    shares.columns = [f'subcategory_cluster_{int(c)}_product_share' for c in shares.columns]
    shares = shares.reset_index()

    counts = cluster_counts.pivot(index='category', columns='combined_cluster', values='product_count').fillna(0)
    counts.columns = [f'subcategory_cluster_{int(c)}_product_count' for c in counts.columns]
    counts = counts.reset_index()

    dominant = (
        cluster_counts.sort_values(['category', 'product_count', 'combined_cluster'], ascending=[True, False, True])
        .drop_duplicates('category')
        [['category', 'combined_cluster', 'product_share']]
        .rename(columns={
            'combined_cluster': 'subcategory_cluster_dominant',
            'product_share': 'subcategory_cluster_dominant_share',
        })
    )
    diversity = (
        cluster_counts.assign(entropy_component=lambda d: -d['product_share'] * np.log(d['product_share'].replace(0, np.nan)))
        .groupby('category', observed=True)
        .agg(
            subcategory_cluster_count=('combined_cluster', 'nunique'),
            subcategory_cluster_entropy=('entropy_component', 'sum'),
        )
        .reset_index()
    )

    features = pd.DataFrame({'category': CAT_ORDER})
    for part in [shares, counts, dominant, diversity]:
        features = features.merge(part, on='category', how='left')

    share_cols = [c for c in features.columns if c.endswith('_product_share')]
    count_cols = [c for c in features.columns if c.endswith('_product_count')]
    features[share_cols + count_cols] = features[share_cols + count_cols].fillna(0)
    features['subcategory_cluster_dominant'] = features['subcategory_cluster_dominant'].fillna(0).astype(int).astype(str)
    features['subcategory_cluster_dominant_share'] = features['subcategory_cluster_dominant_share'].fillna(0)
    features['subcategory_cluster_count'] = features['subcategory_cluster_count'].fillna(0).astype(int)
    features['subcategory_cluster_entropy'] = features['subcategory_cluster_entropy'].fillna(0)

    if output_path is not None:
        features.to_csv(output_path, index=False)
        print(f'Wrote subcategory cluster features: {output_path}')
    return features


subcategory_cluster_features = build_subcategory_cluster_features(
    product,
    SUBCATEGORY_CLUSTER_PATH,
    SUBCATEGORY_CLUSTER_OUTPUT_PATH,
)
print(subcategory_cluster_features.shape)
display(subcategory_cluster_features)



SERVE_TYPE_CODE = {
    'ร้อน': 'hot',
    'เย็น': 'iced',
    'ปั่น': 'blended',
    'ชิ้น': 'piece',
}
SERVE_TYPE_TEMP_SPEARMAN = {
    'hot': -0.685,
    'iced': 0.664,
    'blended': 0.666,
    'piece': 0.025,
}


def build_serve_type_weather_features(product_df, output_path=None):
    """Category-level serve-type mix plus temperature response priors from EDA."""
    prod = product_df.copy()
    prod['serve_type_code'] = prod['serve_type'].map(SERVE_TYPE_CODE).fillna('other')
    counts = (
        prod.groupby(['category', 'serve_type_code'], observed=True)['product_id']
        .nunique()
        .rename('product_count')
        .reset_index()
    )
    totals = counts.groupby('category', observed=True)['product_count'].sum().rename('category_product_count').reset_index()
    counts = counts.merge(totals, on='category', how='left')
    counts['product_share'] = counts['product_count'] / counts['category_product_count'].replace(0, np.nan)
    counts['temp_spearman'] = counts['serve_type_code'].map(SERVE_TYPE_TEMP_SPEARMAN).fillna(0)

    shares = counts.pivot(index='category', columns='serve_type_code', values='product_share').fillna(0)
    shares.columns = [f'serve_type_{c}_product_share' for c in shares.columns]
    counts_wide = counts.pivot(index='category', columns='serve_type_code', values='product_count').fillna(0)
    counts_wide.columns = [f'serve_type_{c}_product_count' for c in counts_wide.columns]

    response = (
        counts.assign(weighted_temp_response=lambda d: d['product_share'] * d['temp_spearman'])
        .groupby('category', observed=True)
        .agg(
            serve_type_count=('serve_type_code', 'nunique'),
            serve_type_temp_response_score=('weighted_temp_response', 'sum'),
            serve_type_temp_positive_share=('product_share', lambda s: s[counts.loc[s.index, 'temp_spearman'].gt(0.2)].sum()),
            serve_type_temp_negative_share=('product_share', lambda s: s[counts.loc[s.index, 'temp_spearman'].lt(-0.2)].sum()),
            serve_type_piece_share=('product_share', lambda s: s[counts.loc[s.index, 'serve_type_code'].eq('piece')].sum()),
        )
        .reset_index()
    )

    features = pd.DataFrame({'category': CAT_ORDER})
    for part in [shares.reset_index(), counts_wide.reset_index(), response]:
        features = features.merge(part, on='category', how='left')
    serve_cols = [c for c in features.columns if c.startswith('serve_type_')]
    features[serve_cols] = features[serve_cols].fillna(0)
    if output_path is not None:
        features.to_csv(output_path, index=False)
        print(f'Wrote serve-type weather features: {output_path}')
    return features


serve_type_weather_features = build_serve_type_weather_features(product, SERVE_TYPE_WEATHER_OUTPUT_PATH)
print(serve_type_weather_features.shape)
display(serve_type_weather_features)


Subcategory cluster file not found: /Users/moi/PROJECT/Expresso/outputs/eda/subcategory_hypothesis/product_group_manual_pca11_clusters.csv; skipping subcategory cluster features
(7, 1)


,category
0,Coffee
1,Tea
2,Bakery
3,Savory Bakery
4,Chocolate & Milk
5,Juice & Smoothie
6,Merchandise


Wrote serve-type weather features: /Users/moi/PROJECT/Expresso/outputs/autogluon_weighted_leakage_safe_horizon/category_serve_type_weather_features.csv
(7, 14)


,category,serve_type_blended_product_share,serve_type_hot_product_share,serve_type_iced_product_share,serve_type_piece_product_share,serve_type_blended_product_count,serve_type_hot_product_count,serve_type_iced_product_count,serve_type_piece_product_count,serve_type_count,serve_type_temp_response_score,serve_type_temp_positive_share,serve_type_temp_negative_share,serve_type_piece_share
0,Coffee,0.176471,0.411765,0.411765,0.0,3.0,7.0,7.0,0.0,3,0.108882,0.588235,0.411765,0.0
1,Tea,0.222222,0.222222,0.555556,0.0,2.0,2.0,5.0,0.0,3,0.364667,0.777778,0.222222,0.0
2,Bakery,0.000000,0.000000,0.000000,1.0,0.0,0.0,0.0,7.0,1,0.025000,0.000000,0.000000,1.0
3,Savory Bakery,0.000000,0.000000,0.000000,1.0,0.0,0.0,0.0,7.0,1,0.025000,0.000000,0.000000,1.0
4,Chocolate & Milk,0.375000,0.250000,0.375000,0.0,3.0,2.0,3.0,0.0,3,0.327500,0.750000,0.250000,0.0
5,Juice & Smoothie,0.800000,0.000000,0.200000,0.0,4.0,0.0,1.0,0.0,2,0.665600,1.000000,0.000000,0.0
6,Merchandise,0.000000,0.000000,0.000000,1.0,0.0,0.0,0.0,7.0,1,0.025000,0.000000,0.000000,1.0


## 3. Known Covariates From Competition Tables

These features are available before the forecast dates: calendar flags, promotions, local events, stockout history summaries, store attributes, and curated events.

In [9]:
def horizon_item_id(store_id, category, horizon):
    category_slug = pd.Series(category).astype(str).str.replace(r'[^A-Za-z0-9]+', '-', regex=True).str.strip('-')
    return pd.Series(store_id).astype(str) + '_' + category_slug + '_' + pd.Series(horizon).astype(str)


def parse_sample_submission(sample_df):
    parsed = sample_df['id'].str.extract(
        r'^(?P<store_id>\d+)_(?P<category>.+)_(?P<forecast_date>\d{4}-\d{2}-\d{2})_(?P<horizon>1d|7d|1m)$'
    )
    if parsed.isna().any().any():
        bad_ids = sample_df.loc[parsed.isna().any(axis=1), 'id'].head(10).tolist()
        raise ValueError(f'Could not parse sample IDs: {bad_ids}')
    parsed['store_id'] = parsed['store_id'].astype(int)
    parsed['forecast_date'] = pd.to_datetime(parsed['forecast_date'])
    parsed['horizon_days'] = parsed['horizon'].map(HORIZON_TO_DAYS).astype(int)
    parsed['decision_date'] = parsed['forecast_date'] - pd.to_timedelta(parsed['horizon_days'], unit='D')
    parsed['effective_decision_date'] = parsed['decision_date'].where(parsed['decision_date'].le(TRAIN_END), TRAIN_END)
    parsed['item_id'] = horizon_item_id(parsed['store_id'], parsed['category'], parsed['horizon'])
    return pd.concat([sample_df[['id']], parsed], axis=1)


def expand_promotions(promo_df, product_df):
    promo_prod = promo_df.merge(product_df[['product_id', 'category']], on='product_id', how='left')
    parts = []
    for row in promo_prod.itertuples(index=False):
        days = pd.date_range(row.start_date, row.end_date, freq='D')
        if len(days) == 0:
            continue
        parts.append(pd.DataFrame({
            'store_id': row.store_id,
            'category': row.category,
            'timestamp': days,
            'product_id': row.product_id,
            'discount_pct': row.discount_pct,
            'email_sent': int(bool(row.email_sent)),
            'social_campaign': int(bool(row.social_campaign)),
            'promo_type': row.promo_type,
        }))
    if not parts:
        return pd.DataFrame(columns=['store_id', 'category', 'timestamp'])
    daily = pd.concat(parts, ignore_index=True)
    out = (
        daily.groupby(['store_id', 'category', 'timestamp'], observed=True)
        .agg(
            active_promo_products=('product_id', 'nunique'),
            max_discount_pct=('discount_pct', 'max'),
            mean_discount_pct=('discount_pct', 'mean'),
            email_sent=('email_sent', 'max'),
            social_campaign=('social_campaign', 'max'),
            promo_type_count=('promo_type', 'nunique'),
        )
        .reset_index()
    )
    out['promo_channel_count'] = out[['email_sent', 'social_campaign']].sum(axis=1)
    out['promo_discount_intensity'] = out['active_promo_products'] * out['mean_discount_pct'].fillna(0)
    out['has_deep_discount'] = out['max_discount_pct'].ge(25).astype(int)
    return out


LOCAL_EVENT_TYPES = [
    'book_fair', 'concert', 'convention', 'cultural', 'food_festival',
    'market', 'music_festival', 'sports',
]
LOCAL_EVENT_KEYWORDS = {
    'coffee': ['กาแฟ', 'coffee'],
    'food': ['อาหาร', 'street food', 'ของหวาน', 'เบียร์', 'คราฟท์', 'food', 'dessert'],
    'music': ['ดนตรี', 'คอนเสิร์ต', 'แจ๊ส', 'busking', 'music', 'concert', 'festival'],
    'student': ['นักศึกษา', 'startup', 'job fair', 'กีฬาสี'],
    'book': ['หนังสือ', 'book'],
    'auto': ['auto', 'motor', 'รถ'],
    'religious': ['งานบวช', 'วัด', 'สวดมนต์'],
    'night': ['night', 'กลางคืน', 'busking', 'แจ๊ส'],
    'sport': ['กีฬา', 'โยคะ', 'จักรยาน', 'sports'],
    'market': ['ตลาด', 'market', 'ถนนคนเดิน'],
}
EVENT_TYPE_STORE_RELEVANCE = {
    'concert': {'mall': 1.0, 'tourist': 0.9, 'transit': 0.8, 'urban_residential': 0.7, 'university': 0.7, 'office': 0.4, 'hospital': 0.3, 'gas_station': 0.3},
    'music_festival': {'mall': 1.0, 'tourist': 1.0, 'transit': 0.8, 'urban_residential': 0.7, 'university': 0.7, 'office': 0.4, 'hospital': 0.3, 'gas_station': 0.3},
    'food_festival': {'mall': 0.9, 'tourist': 1.0, 'urban_residential': 0.8, 'transit': 0.7, 'university': 0.6, 'office': 0.5, 'hospital': 0.4, 'gas_station': 0.4},
    'market': {'urban_residential': 1.0, 'tourist': 0.8, 'transit': 0.7, 'university': 0.7, 'mall': 0.5, 'office': 0.4, 'hospital': 0.4, 'gas_station': 0.4},
    'book_fair': {'university': 1.0, 'mall': 0.8, 'tourist': 0.6, 'urban_residential': 0.6, 'office': 0.5, 'transit': 0.5, 'hospital': 0.3, 'gas_station': 0.2},
    'convention': {'office': 1.0, 'mall': 0.9, 'transit': 0.8, 'tourist': 0.7, 'university': 0.7, 'urban_residential': 0.5, 'hospital': 0.4, 'gas_station': 0.3},
    'sports': {'university': 0.9, 'urban_residential': 0.8, 'transit': 0.7, 'tourist': 0.6, 'mall': 0.5, 'gas_station': 0.5, 'office': 0.3, 'hospital': 0.3},
    'cultural': {'tourist': 1.0, 'urban_residential': 0.8, 'mall': 0.7, 'university': 0.6, 'transit': 0.6, 'gas_station': 0.4, 'hospital': 0.3, 'office': 0.3},
}
EVENT_TYPE_CATEGORY_RELEVANCE = {
    'concert': {'Coffee': 1.0, 'Tea': 0.9, 'Chocolate & Milk': 0.8, 'Juice & Smoothie': 0.7, 'Bakery': 0.6, 'Savory Bakery': 0.6, 'Merchandise': 0.2},
    'music_festival': {'Coffee': 1.0, 'Tea': 0.9, 'Chocolate & Milk': 0.8, 'Juice & Smoothie': 0.8, 'Bakery': 0.6, 'Savory Bakery': 0.7, 'Merchandise': 0.2},
    'food_festival': {'Savory Bakery': 1.0, 'Bakery': 0.9, 'Juice & Smoothie': 0.8, 'Coffee': 0.7, 'Tea': 0.6, 'Chocolate & Milk': 0.6, 'Merchandise': 0.1},
    'market': {'Coffee': 0.8, 'Tea': 0.7, 'Bakery': 0.8, 'Savory Bakery': 0.8, 'Juice & Smoothie': 0.7, 'Chocolate & Milk': 0.5, 'Merchandise': 0.2},
    'book_fair': {'Coffee': 1.0, 'Tea': 0.8, 'Bakery': 0.6, 'Chocolate & Milk': 0.5, 'Savory Bakery': 0.4, 'Juice & Smoothie': 0.4, 'Merchandise': 0.3},
    'convention': {'Coffee': 1.0, 'Tea': 0.7, 'Savory Bakery': 0.8, 'Bakery': 0.6, 'Chocolate & Milk': 0.4, 'Juice & Smoothie': 0.3, 'Merchandise': 0.2},
    'sports': {'Juice & Smoothie': 1.0, 'Coffee': 0.8, 'Tea': 0.7, 'Savory Bakery': 0.5, 'Bakery': 0.4, 'Chocolate & Milk': 0.4, 'Merchandise': 0.1},
    'cultural': {'Coffee': 0.8, 'Tea': 0.8, 'Bakery': 0.7, 'Juice & Smoothie': 0.7, 'Chocolate & Milk': 0.5, 'Savory Bakery': 0.5, 'Merchandise': 0.2},
}
EVENT_TYPE_BASE_INTENSITY = {
    'music_festival': 1.20, 'concert': 1.10, 'food_festival': 1.05, 'convention': 0.95,
    'market': 0.85, 'book_fair': 0.80, 'sports': 0.75, 'cultural': 0.70,
}


def normalize_event_type(value):
    return str(value).strip().lower().replace(' ', '_')


def event_keyword_flags(event_names):
    text = ' | '.join(str(x).lower() for x in event_names)
    return {
        f'kaggle_event_kw_{key}': int(any(token.lower() in text for token in tokens))
        for key, tokens in LOCAL_EVENT_KEYWORDS.items()
    }


def build_local_event_base(local_event_df):
    events = local_event_df.rename(columns={'date': 'timestamp'}).copy()
    events['timestamp'] = pd.to_datetime(events['timestamp'])
    events['event_type_norm'] = events['event_type'].map(normalize_event_type)
    return events


def build_local_event_store_day_features(local_event_df):
    events = build_local_event_base(local_event_df)
    rows = []
    for (store_id, timestamp), group in events.groupby(['store_id', 'timestamp'], observed=True):
        types = sorted(set(group['event_type_norm']))
        names = group['event_name'].astype(str).tolist()
        row = {
            'store_id': store_id,
            'timestamp': timestamp,
            'kaggle_local_event_count': len(group),
            'kaggle_local_event_type_count': len(types),
            'kaggle_event_intensity_sum': sum(EVENT_TYPE_BASE_INTENSITY.get(t, 0.5) for t in group['event_type_norm']),
            'kaggle_event_intensity_max': max(EVENT_TYPE_BASE_INTENSITY.get(t, 0.5) for t in group['event_type_norm']),
            'kaggle_event_name_length_mean': float(np.mean([len(name) for name in names])) if names else 0.0,
            'kaggle_has_music_event': int(any(t in {'music_festival', 'concert'} for t in types)),
            'kaggle_has_market_event': int('market' in types),
            'kaggle_has_cultural_event': int('cultural' in types),
        }
        for event_type in LOCAL_EVENT_TYPES:
            row[f'kaggle_event_type_{event_type}'] = int(event_type in types)
        row.update(event_keyword_flags(names))
        rows.append(row)
    if not rows:
        cols = ['store_id', 'timestamp', 'kaggle_local_event_count']
        return pd.DataFrame(columns=cols)
    return pd.DataFrame(rows)


def build_local_event_category_day_features(local_event_df, store_df, categories):
    events = build_local_event_base(local_event_df).merge(store_df[['store_id', 'neighborhood_type']], on='store_id', how='left')
    rows = []
    for event in events.to_dict('records'):
        event_type = event['event_type_norm']
        store_type = event['neighborhood_type']
        store_weight = EVENT_TYPE_STORE_RELEVANCE.get(event_type, {}).get(store_type, 0.35)
        base_intensity = EVENT_TYPE_BASE_INTENSITY.get(event_type, 0.50)
        kw = event_keyword_flags([event['event_name']])
        for category in categories:
            cat_weight = EVENT_TYPE_CATEGORY_RELEVANCE.get(event_type, {}).get(category, 0.35)
            row = {
                'store_id': event['store_id'],
                'category': category,
                'timestamp': event['timestamp'],
                'kaggle_event_category_match_count': 1 if cat_weight >= 0.7 else 0,
                'kaggle_event_store_relevance_max': store_weight,
                'kaggle_event_category_relevance_max': cat_weight,
                'kaggle_event_relevance_score': base_intensity * store_weight * cat_weight,
            }
            for event_type_name in LOCAL_EVENT_TYPES:
                row[f'kaggle_event_type_{event_type_name}_category_score'] = (
                    base_intensity * store_weight * cat_weight if event_type == event_type_name else 0.0
                )
            for key, value in kw.items():
                row[f'{key}_category_score'] = value * store_weight * cat_weight
            rows.append(row)
    if not rows:
        return pd.DataFrame(columns=['store_id', 'category', 'timestamp'])
    raw = pd.DataFrame(rows)
    agg = {col: 'sum' for col in raw.columns if col.endswith('_score') or col == 'kaggle_event_category_match_count'}
    agg.update({
        'kaggle_event_store_relevance_max': 'max',
        'kaggle_event_category_relevance_max': 'max',
    })
    return raw.groupby(['store_id', 'category', 'timestamp'], observed=True).agg(agg).reset_index()


def add_local_event_window_features(store_day_features, store_df, start, end):
    grid = pd.MultiIndex.from_product(
        [sorted(store_df['store_id'].unique()), pd.date_range(start, end, freq='D')],
        names=['store_id', 'timestamp'],
    ).to_frame(index=False)
    out = grid.merge(store_day_features, on=['store_id', 'timestamp'], how='left')
    event_cols = [c for c in out.columns if c.startswith('kaggle_')]
    out[event_cols] = out[event_cols].fillna(0)
    out = out.sort_values(['store_id', 'timestamp'])
    grouped = out.groupby('store_id', observed=True)
    out['kaggle_local_event_count_lag1'] = grouped['kaggle_local_event_count'].shift(1).fillna(0)
    out['kaggle_local_event_count_lead1'] = grouped['kaggle_local_event_count'].shift(-1).fillna(0)
    out['kaggle_local_event_window_3d'] = (
        out['kaggle_local_event_count_lag1'] + out['kaggle_local_event_count'] + out['kaggle_local_event_count_lead1']
    )

    pieces = []
    for _, part in out.groupby('store_id', observed=True, sort=False):
        part = part.copy()
        event_dates = part.loc[part['kaggle_local_event_count'].gt(0), 'timestamp'].to_numpy(dtype='datetime64[D]')
        days = part['timestamp'].to_numpy(dtype='datetime64[D]')
        if len(event_dates) == 0:
            part['kaggle_days_since_local_event'] = 99
            part['kaggle_days_until_local_event'] = 99
        else:
            prev_idx = np.searchsorted(event_dates, days, side='right') - 1
            next_idx = np.searchsorted(event_dates, days, side='left')
            prev_days = np.where(prev_idx >= 0, (days - event_dates[np.clip(prev_idx, 0, len(event_dates)-1)]).astype('timedelta64[D]').astype(int), 99)
            next_days = np.where(next_idx < len(event_dates), (event_dates[np.clip(next_idx, 0, len(event_dates)-1)] - days).astype('timedelta64[D]').astype(int), 99)
            part['kaggle_days_since_local_event'] = np.clip(prev_days, 0, 30)
            part['kaggle_days_until_local_event'] = np.clip(next_days, 0, 30)
        pieces.append(part)
    out = pd.concat(pieces, ignore_index=True)
    return out


def build_stockout_asof_features(inventory_df, product_df, store_df, categories, start, cutoff):
    inv_prod = inventory_df[inventory_df['date'].le(cutoff)].merge(product_df[['product_id', 'category']], on='product_id', how='left')
    daily = (
        inv_prod.groupby(['store_id', 'category', 'date'], observed=True)
        .agg(
            any_stockout=('is_stockout', 'max'),
            stockout_sku_share=('is_stockout', 'mean'),
            closing_stock=('closing_stock', 'sum'),
        )
        .reset_index()
    )
    idx = pd.MultiIndex.from_product(
        [sorted(store_df['store_id'].unique()), categories, pd.date_range(start, cutoff, freq='D')],
        names=['store_id', 'category', 'effective_decision_date'],
    )
    asof = idx.to_frame(index=False).merge(
        daily.rename(columns={'date': 'effective_decision_date'}),
        on=['store_id', 'category', 'effective_decision_date'],
        how='left',
    )
    for col in ['any_stockout', 'stockout_sku_share', 'closing_stock']:
        asof[col] = pd.to_numeric(asof[col], errors='coerce').fillna(0)
    asof = asof.sort_values(['store_id', 'category', 'effective_decision_date'])
    grouped = asof.groupby(['store_id', 'category'], observed=True)
    asof['hist_stockout_rate'] = grouped['any_stockout'].transform(lambda s: s.shift(1).expanding(min_periods=1).mean()).fillna(0)
    asof['hist_stockout_sku_share'] = grouped['stockout_sku_share'].transform(lambda s: s.shift(1).expanding(min_periods=1).mean()).fillna(0)
    asof['hist_mean_closing_stock'] = grouped['closing_stock'].transform(lambda s: s.shift(1).expanding(min_periods=1).mean()).fillna(0)
    asof['stockout_rate_7d'] = grouped['any_stockout'].transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean()).fillna(0)
    asof['stockout_rate_28d'] = grouped['any_stockout'].transform(lambda s: s.shift(1).rolling(28, min_periods=1).mean()).fillna(0)
    asof['stockout_sku_share_7d'] = grouped['stockout_sku_share'].transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean()).fillna(0)
    asof['stockout_sku_share_28d'] = grouped['stockout_sku_share'].transform(lambda s: s.shift(1).rolling(28, min_periods=1).mean()).fillna(0)
    asof['closing_stock_mean_7d'] = grouped['closing_stock'].transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean()).fillna(0)
    asof['closing_stock_mean_28d'] = grouped['closing_stock'].transform(lambda s: s.shift(1).rolling(28, min_periods=1).mean()).fillna(0)

    def stockout_recency(part):
        dates = part['effective_decision_date'].to_numpy(dtype='datetime64[D]')
        stockout_dates = part.loc[part['any_stockout'].gt(0), 'effective_decision_date'].to_numpy(dtype='datetime64[D]')
        if len(stockout_dates) == 0:
            return pd.Series(np.full(len(part), 99), index=part.index)
        prev_idx = np.searchsorted(stockout_dates, dates, side='left') - 1
        prev_days = np.where(
            prev_idx >= 0,
            (dates - stockout_dates[np.clip(prev_idx, 0, len(stockout_dates) - 1)]).astype('timedelta64[D]').astype(int),
            99,
        )
        return pd.Series(np.clip(prev_days, 0, 99), index=part.index)

    asof['days_since_stockout'] = grouped.apply(stockout_recency).reset_index(level=[0, 1], drop=True).fillna(99)
    return asof[['store_id', 'category', 'effective_decision_date'] + STOCKOUT_ASOF_FEATURES]


sample_parsed = parse_sample_submission(sample)
promo_features = expand_promotions(promotion, product)
local_event_store_day_raw = build_local_event_store_day_features(local_event)
local_event_features = add_local_event_window_features(local_event_store_day_raw, store, pd.Timestamp('2023-01-01'), FORECAST_END)
local_event_category_features = build_local_event_category_day_features(local_event, store, CAT_ORDER)
stockout_asof = build_stockout_asof_features(inventory, product, store, CAT_ORDER, pd.Timestamp('2023-01-01'), TRAIN_END)

print(promo_features.shape, local_event_features.shape, local_event_category_features.shape, stockout_asof.shape)
sample_parsed.head()


(39901, 12) (14620, 33) (9597, 25) (93800, 13)


/var/folders/zm/l71z21jd7gg4w4y6px1y79k80000gn/T/ipykernel_8187/3949508008.py:270: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  asof['days_since_stockout'] = grouped.apply(stockout_recency).reset_index(level=[0, 1], drop=True).fillna(99)


,id,store_id,category,forecast_date,horizon,horizon_days,decision_date,effective_decision_date,item_id
0,1_Bakery_2024-11-01_1d,1,Bakery,2024-11-01,1d,1,2024-10-31,2024-10-31,1_Bakery_1d
1,1_Bakery_2024-11-01_1m,1,Bakery,2024-11-01,1m,30,2024-10-02,2024-10-02,1_Bakery_1m
2,1_Bakery_2024-11-01_7d,1,Bakery,2024-11-01,7d,7,2024-10-25,2024-10-25,1_Bakery_7d
3,1_Bakery_2024-11-02_1d,1,Bakery,2024-11-02,1d,1,2024-11-01,2024-10-31,1_Bakery_1d
4,1_Bakery_2024-11-02_1m,1,Bakery,2024-11-02,1m,30,2024-10-03,2024-10-03,1_Bakery_1m


In [10]:
def build_special_events():
    rows = [
        # Forecast-window Bangkok demand shocks already researched in this project.
        ('awakening_bangkok_2024', 'Awakening Bangkok', '2024-11-08', '2024-11-17', ['Coffee', 'Tea'], ['tourist', 'urban_residential'], ['mall', 'transit'], 0.90),
        ('loy_krathong_2024', 'Loy Krathong', '2024-11-15', '2024-11-15', ['Juice & Smoothie', 'Bakery'], ['tourist', 'urban_residential', 'mall'], ['transit', 'university'], 1.00),
        ('motor_expo_2024', 'Motor Expo 2024', '2024-11-29', '2024-12-10', ['Savory Bakery', 'Coffee'], ['gas_station', 'transit'], ['tourist', 'urban_residential', 'mall'], 0.95),
        ('red_cross_fair_2024', 'Red Cross Fair', '2024-12-11', '2024-12-22', ['ALL'], ['tourist', 'urban_residential', 'mall'], ['transit', 'office', 'university', 'hospital'], 1.15),
        ('christmas_2024', 'Christmas', '2024-12-25', '2024-12-25', ['Merchandise', 'Chocolate & Milk'], ['mall', 'tourist', 'urban_residential'], ['office', 'transit'], 1.00),
        ('new_year_countdown_2024', 'New Year Countdown', '2024-12-31', '2024-12-31', ['ALL'], ['tourist', 'mall', 'transit', 'urban_residential'], ['gas_station', 'office'], 1.20),
        # Optional university/commencement style signals. Keep these extensible; many ceremonies vary by university and venue.
        ('graduation_university_proxy_2024', 'University graduation proxy', '2024-10-01', '2024-10-31', ['Coffee', 'Bakery', 'Merchandise'], ['university'], ['transit', 'mall'], 0.50),
    ]
    expanded = []
    for event_key, event_name, start, end, categories, strong_types, secondary_types, base_weight in rows:
        for ts in pd.date_range(start, end, freq='D'):
            expanded.append({
                'event_key': event_key,
                'event_name': event_name,
                'timestamp': ts,
                'impacted_categories': categories,
                'strong_store_types': strong_types,
                'secondary_store_types': secondary_types,
                'base_weight': base_weight,
            })
    return pd.DataFrame(expanded)


def special_event_features(calendar_df, store_df):
    grid = []
    for row in calendar_df.to_dict('records'):
        for store_row in store_df[['store_id', 'neighborhood_type']].to_dict('records'):
            for category in CAT_ORDER:
                cat_match = 'ALL' in row['impacted_categories'] or category in row['impacted_categories']
                stype = store_row['neighborhood_type']
                if 'ALL' in row['strong_store_types'] or stype in row['strong_store_types']:
                    st_weight = 1.0
                elif 'ALL' in row['secondary_store_types'] or stype in row['secondary_store_types']:
                    st_weight = 0.65
                else:
                    st_weight = 0.25
                grid.append({
                    'store_id': store_row['store_id'],
                    'category': category,
                    'timestamp': row['timestamp'],
                    'special_event_count': 1,
                    'special_event_category_match': int(cat_match),
                    'special_event_store_weight': st_weight,
                    'special_event_score': float(row['base_weight']) * st_weight if cat_match else 0.0,
                })
    out = pd.DataFrame(grid)
    return (
        out.groupby(['store_id', 'category', 'timestamp'], observed=True)
        .agg(
            special_event_count=('special_event_count', 'sum'),
            special_event_category_match=('special_event_category_match', 'max'),
            special_event_store_weight=('special_event_store_weight', 'max'),
            special_event_score=('special_event_score', 'sum'),
        )
        .reset_index()
    )


special_calendar = build_special_events()
special_features = special_event_features(special_calendar, store)
special_calendar.head()

,event_key,event_name,timestamp,impacted_categories,strong_store_types,secondary_store_types,base_weight
0,awakening_bangkok_2024,Awakening Bangkok,2024-11-08,"[Coffee, Tea]","[tourist, urban_residential]","[mall, transit]",0.9
1,awakening_bangkok_2024,Awakening Bangkok,2024-11-09,"[Coffee, Tea]","[tourist, urban_residential]","[mall, transit]",0.9
2,awakening_bangkok_2024,Awakening Bangkok,2024-11-10,"[Coffee, Tea]","[tourist, urban_residential]","[mall, transit]",0.9
3,awakening_bangkok_2024,Awakening Bangkok,2024-11-11,"[Coffee, Tea]","[tourist, urban_residential]","[mall, transit]",0.9
4,awakening_bangkok_2024,Awakening Bangkok,2024-11-12,"[Coffee, Tea]","[tourist, urban_residential]","[mall, transit]",0.9


## 4. External Weather, Salary, Holiday, And Oil Features

The cells below cache external data under `outputs/autogluon_external/` so reruns are fast and references remain traceable.

In [11]:
def fetch_json_url(url, timeout=60):
    with urllib.request.urlopen(url, timeout=timeout) as response:
        return json.loads(response.read().decode('utf-8'))


def fetch_open_meteo_weather(start, end, mode='actual_archive'):
    cache_path = OUTPUT_DIR / f'open_meteo_bangkok_{start:%Y%m%d}_{end:%Y%m%d}_{mode}.csv'
    if cache_path.exists():
        return pd.read_csv(cache_path, parse_dates=['timestamp'])

    dates = pd.date_range(start, end, freq='D')
    if mode == 'climatology':
        # Approximate Bangkok monthly normals used only as a leakage-safe fallback.
        normals = pd.DataFrame({
            'month': list(range(1, 13)),
            'temperature_2m_mean': [26.7, 28.2, 29.7, 30.7, 30.1, 29.5, 29.1, 28.9, 28.6, 28.2, 27.5, 26.3],
            'temperature_2m_max': [32.0, 33.0, 34.3, 35.4, 34.6, 33.6, 33.2, 32.9, 32.7, 32.5, 32.1, 31.3],
            'temperature_2m_min': [21.8, 23.4, 25.0, 26.1, 26.0, 25.8, 25.5, 25.4, 25.1, 24.6, 23.4, 21.7],
            'relative_humidity_2m_mean': [66, 69, 72, 73, 76, 77, 78, 79, 81, 79, 73, 67],
            'precipitation_sum': [9, 20, 40, 91, 248, 157, 175, 219, 333, 190, 40, 11],
            'precipitation_hours': [1, 2, 3, 6, 14, 13, 14, 16, 20, 14, 5, 1],
        })
        out = pd.DataFrame({'timestamp': dates})
        out['month'] = out['timestamp'].dt.month
        out = out.merge(normals, on='month', how='left').drop(columns=['month'])
        out['precipitation_sum'] = out['precipitation_sum'] / out['timestamp'].dt.days_in_month
        out['precipitation_hours'] = out['precipitation_hours'] / out['timestamp'].dt.days_in_month
    else:
        params = {
            'latitude': 13.7563,
            'longitude': 100.5018,
            'start_date': start.strftime('%Y-%m-%d'),
            'end_date': end.strftime('%Y-%m-%d'),
            'daily': ','.join([
                'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min',
                'relative_humidity_2m_mean', 'precipitation_sum', 'precipitation_hours',
            ]),
            'timezone': 'Asia/Bangkok',
        }
        url = 'https://archive-api.open-meteo.com/v1/archive?' + urllib.parse.urlencode(params)
        payload = fetch_json_url(url)
        out = pd.DataFrame(payload['daily']).rename(columns={'time': 'timestamp'})
        out['timestamp'] = pd.to_datetime(out['timestamp'])

    out['rain_flag'] = out['precipitation_sum'].gt(0).astype(int)
    out['heavy_rain_flag'] = out['precipitation_sum'].ge(10).astype(int)
    out['hot_day_flag'] = out['temperature_2m_max'].ge(33).astype(int)
    out['cool_day_flag'] = out['temperature_2m_mean'].le(26.5).astype(int)
    out['humid_day_flag'] = out['relative_humidity_2m_mean'].ge(80).astype(int)
    out.to_csv(cache_path, index=False)
    return out


weather = fetch_open_meteo_weather(pd.Timestamp('2023-01-01'), FORECAST_END, WEATHER_MODE)
weather.tail()

,timestamp,temperature_2m_mean,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum,precipitation_hours,rain_flag,heavy_rain_flag,hot_day_flag,cool_day_flag,humid_day_flag
726,2024-12-27,26.3,31.3,21.7,67,0.354839,0.032258,1,0,0,1,0
727,2024-12-28,26.3,31.3,21.7,67,0.354839,0.032258,1,0,0,1,0
728,2024-12-29,26.3,31.3,21.7,67,0.354839,0.032258,1,0,0,1,0
729,2024-12-30,26.3,31.3,21.7,67,0.354839,0.032258,1,0,0,1,0
730,2024-12-31,26.3,31.3,21.7,67,0.354839,0.032258,1,0,0,1,0


In [12]:
def salary_and_public_calendar_features(start, end):
    df = pd.DataFrame({'timestamp': pd.date_range(start, end, freq='D')})

    # Government salary/pension payment dates. For 2024, civil servants shifted to two monthly rounds;
    # pensioners have a separate monthly payment date. These are encoded as known calendar shocks.
    salary_dates_2024 = pd.to_datetime([
        '2024-01-16', '2024-01-26', '2024-02-16', '2024-02-23', '2024-03-15', '2024-03-26',
        '2024-04-12', '2024-04-25', '2024-05-16', '2024-05-28', '2024-06-14', '2024-06-25',
        '2024-07-16', '2024-07-25', '2024-08-16', '2024-08-27', '2024-09-16', '2024-09-25',
        '2024-10-16', '2024-10-28', '2024-11-15', '2024-11-26', '2024-12-16', '2024-12-24',
    ])
    pension_dates_2024 = pd.to_datetime([
        '2024-01-24', '2024-02-21', '2024-03-22', '2024-04-23', '2024-05-24', '2024-06-21',
        '2024-07-23', '2024-08-23', '2024-09-23', '2024-10-24', '2024-11-22', '2024-12-20',
    ])

    # Generic private-sector salary effect near month end; keep broad because office payroll dates vary.
    df['is_month_end_salary_window'] = df['timestamp'].dt.day.between(25, 31).astype(int)
    df['is_civil_servant_payday'] = df['timestamp'].isin(salary_dates_2024).astype(int)
    df['is_pension_payday'] = df['timestamp'].isin(pension_dates_2024).astype(int)
    df['is_any_salary_payday'] = df[['is_month_end_salary_window', 'is_civil_servant_payday', 'is_pension_payday']].max(axis=1)

    paydays = sorted(set(salary_dates_2024.tolist() + pension_dates_2024.tolist()))
    payday_arr = np.array(paydays, dtype='datetime64[D]')
    day_arr = df['timestamp'].values.astype('datetime64[D]')
    if len(payday_arr):
        prev_idx = np.searchsorted(payday_arr, day_arr, side='right') - 1
        next_idx = np.searchsorted(payday_arr, day_arr, side='left')
        prev_days = np.where(prev_idx >= 0, (day_arr - payday_arr[np.clip(prev_idx, 0, len(payday_arr)-1)]).astype('timedelta64[D]').astype(int), 99)
        next_days = np.where(next_idx < len(payday_arr), (payday_arr[np.clip(next_idx, 0, len(payday_arr)-1)] - day_arr).astype('timedelta64[D]').astype(int), 99)
    else:
        prev_days = np.full(len(df), 99)
        next_days = np.full(len(df), 99)
    df['days_since_gov_payday'] = np.clip(prev_days, 0, 31)
    df['days_until_gov_payday'] = np.clip(next_days, 0, 31)
    df['post_payday_3d'] = df['days_since_gov_payday'].between(0, 3).astype(int)
    df['pre_payday_3d'] = df['days_until_gov_payday'].between(0, 3).astype(int)

    # Major national holidays in forecast window and useful historical dates are also present in DATE_DIM,
    # but explicit flags keep the source visible and make extension easy.
    extra_holidays = pd.to_datetime([
        '2024-01-01', '2024-02-24', '2024-04-06', '2024-04-13', '2024-04-14', '2024-04-15', '2024-04-16',
        '2024-05-01', '2024-05-04', '2024-05-22', '2024-06-03', '2024-07-20', '2024-07-28', '2024-08-12',
        '2024-10-13', '2024-10-23', '2024-12-05', '2024-12-10', '2024-12-31',
    ])
    df['external_public_holiday_flag'] = df['timestamp'].isin(extra_holidays).astype(int)
    df['new_year_countdown_flag'] = df['timestamp'].eq(pd.Timestamp('2024-12-31')).astype(int)
    return df


salary_calendar = salary_and_public_calendar_features(pd.Timestamp('2023-01-01'), FORECAST_END)
salary_calendar.tail(12)

,timestamp,is_month_end_salary_window,is_civil_servant_payday,is_pension_payday,is_any_salary_payday,days_since_gov_payday,days_until_gov_payday,post_payday_3d,pre_payday_3d,external_public_holiday_flag,new_year_countdown_flag
719,2024-12-20,0,0,1,1,0,0,1,1,0,0
720,2024-12-21,0,0,0,0,1,3,1,1,0,0
721,2024-12-22,0,0,0,0,2,2,1,1,0,0
722,2024-12-23,0,0,0,0,3,1,1,1,0,0
723,2024-12-24,0,1,0,1,0,0,1,1,0,0
724,2024-12-25,1,0,0,1,1,31,1,0,0,0
725,2024-12-26,1,0,0,1,2,31,1,0,0,0
726,2024-12-27,1,0,0,1,3,31,1,0,0,0
727,2024-12-28,1,0,0,1,4,31,0,0,0,0
728,2024-12-29,1,0,0,1,5,31,0,0,0,0


In [13]:
def ptt_oil_one_day(day, language='en', retries=3, sleep_seconds=0.75):
    body = f'''<?xml version="1.0" encoding="utf-8"?>
<soap:Envelope xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
  <soap:Body>
    <GetOilPrice xmlns="http://www.pttor.com">
      <Language>{language}</Language>
      <DD>{day.day}</DD>
      <MM>{day.month}</MM>
      <YYYY>{day.year}</YYYY>
    </GetOilPrice>
  </soap:Body>
</soap:Envelope>'''.encode('utf-8')
    req = urllib.request.Request(
        'https://orapiweb.pttor.com/oilservice/OilPrice.asmx',
        data=body,
        headers={
            'Content-Type': 'text/xml; charset=utf-8',
            'SOAPAction': '"https://orapiweb.pttor.com/GetOilPrice"',
        },
        method='POST',
    )
    last_exc = None
    for attempt in range(1, retries + 1):
        try:
            with urllib.request.urlopen(req, timeout=30) as response:
                xml_text = response.read().decode('utf-8')
            root = ET.fromstring(xml_text)
            result_text = None
            for elem in root.iter():
                if elem.tag.endswith('GetOilPriceResult'):
                    result_text = elem.text
                    break
            if not result_text:
                return pd.DataFrame()
            inner = ET.fromstring(html.unescape(result_text))
            rows = []
            for fuel in inner.findall('.//FUEL'):
                rec = {child.tag: child.text for child in fuel}
                rows.append(rec)
            out = pd.DataFrame(rows)
            if out.empty:
                return out
            out['query_date'] = pd.Timestamp(day).normalize()
            out['PRICE'] = pd.to_numeric(out['PRICE'], errors='coerce')
            out['PRICE_DATE'] = pd.to_datetime(out['PRICE_DATE'], errors='coerce')
            return out
        except Exception as exc:
            last_exc = exc
            if attempt < retries:
                time.sleep(sleep_seconds * attempt)
    raise last_exc


def fetch_ptt_oil_prices(start, end, enabled=True):
    final_cache_path = OUTPUT_DIR / f'ptt_oil_{start:%Y%m%d}_{end:%Y%m%d}.csv'
    raw_success_path = OUTPUT_DIR / f'ptt_oil_raw_success_{start:%Y%m%d}_{end:%Y%m%d}.csv'
    fail_log_path = OUTPUT_DIR / f'ptt_oil_failed_dates_{start:%Y%m%d}_{end:%Y%m%d}.csv'

    if final_cache_path.exists():
        return pd.read_csv(final_cache_path, parse_dates=['timestamp'])
    if not enabled:
        return pd.DataFrame({'timestamp': pd.date_range(start, end, freq='D')})

    if raw_success_path.exists():
        raw_success = pd.read_csv(raw_success_path, parse_dates=['query_date', 'PRICE_DATE'])
    else:
        raw_success = pd.DataFrame()

    fetched_dates = set()
    if not raw_success.empty and 'query_date' in raw_success.columns:
        fetched_dates = set(pd.to_datetime(raw_success['query_date']).dt.normalize())

    failures = []
    for day in pd.date_range(start, end, freq='D'):
        day = pd.Timestamp(day).normalize()
        if day in fetched_dates:
            continue
        try:
            raw = ptt_oil_one_day(day)
            if not raw.empty:
                raw_success = pd.concat([raw_success, raw], ignore_index=True)
                raw_success = raw_success.drop_duplicates(['query_date', 'PRODUCT'], keep='last')
                raw_success.sort_values(['query_date', 'PRODUCT']).to_csv(raw_success_path, index=False)
                fetched_dates.add(day)
        except Exception as exc:
            print(f'PTT oil fetch failed for {day.date()}: {exc}')
            failures.append({'date': day.date().isoformat(), 'error': str(exc)})
        time.sleep(0.03)

    if failures:
        pd.DataFrame(failures).to_csv(fail_log_path, index=False)

    if raw_success.empty:
        out = pd.DataFrame({'timestamp': pd.date_range(start, end, freq='D')})
    else:
        pivot = raw_success.pivot_table(index='query_date', columns='PRODUCT', values='PRICE', aggfunc='last').reset_index()
        out = pivot.rename(columns={'query_date': 'timestamp'}).drop_duplicates('timestamp').sort_values('timestamp')
        out = pd.DataFrame({'timestamp': pd.date_range(start, end, freq='D')}).merge(out, on='timestamp', how='left')
        out = out.ffill().bfill()

    rename = {
        'Gasohol 95': 'ptt_gasohol95',
        'Gasohol 91': 'ptt_gasohol91',
        'Gasohol E20': 'ptt_gasohol_e20',
        'Diesel': 'ptt_diesel',
        'Premium Diesel': 'ptt_premium_diesel',
        'Gasoline 95': 'ptt_gasoline95',
    }
    out = out.rename(columns={k: v for k, v in rename.items() if k in out.columns})
    keep = ['timestamp'] + [v for v in rename.values() if v in out.columns]
    out = out[keep]
    for col in keep:
        if col != 'timestamp':
            out[col] = pd.to_numeric(out[col], errors='coerce')
    out.to_csv(final_cache_path, index=False)
    return out


# Strict leakage-safe notebook: do not load cached or actual future PTT oil prices.
ptt_oil = pd.DataFrame({'timestamp': pd.date_range(pd.Timestamp('2023-01-01'), FORECAST_END, freq='D')})
ptt_oil.tail()


,timestamp
726,2024-12-27
727,2024-12-28
728,2024-12-29
729,2024-12-30
730,2024-12-31


In [14]:
def parse_bangchak_historical_html(html_text):
    rows = []
    for tr in re.findall(r'<tr[^>]*>(.*?)</tr>', html_text, flags=re.S | re.I):
        date_match = re.search(r'<th[^>]*scope=["\']row["\'][^>]*>(.*?)</th>', tr, flags=re.S | re.I)
        if not date_match:
            continue
        date_text = re.sub(r'<[^>]+>', '', date_match.group(1)).strip()
        values = {'timestamp': date_text}
        for title, value in re.findall(r'<td[^>]*title=["\']([^"\']+)["\'][^>]*>(.*?)</td>', tr, flags=re.S | re.I):
            clean_key = re.sub(r'[^a-z0-9]+', '_', title.lower()).strip('_')
            clean_value = re.sub(r'<[^>]+>', '', value).strip()
            values[f'bangchak_{clean_key}'] = pd.to_numeric(clean_value, errors='coerce')
        if len(values) > 1:
            rows.append(values)
    return pd.DataFrame(rows)


def fetch_bangchak_oil_prices(years=(2023, 2024), enabled=True):
    final_cache_path = OUTPUT_DIR / f'bangchak_oil_{"_".join(map(str, years))}.csv'
    raw_cache_path = OUTPUT_DIR / f'bangchak_oil_raw_{"_".join(map(str, years))}.csv'
    fail_log_path = OUTPUT_DIR / f'bangchak_oil_failed_years_{"_".join(map(str, years))}.csv'
    html_dir = OUTPUT_DIR / 'raw_html'
    html_dir.mkdir(parents=True, exist_ok=True)

    if not enabled:
        return pd.DataFrame(columns=['timestamp'])

    raw = pd.read_csv(raw_cache_path, parse_dates=['timestamp']) if raw_cache_path.exists() else pd.DataFrame()
    cached_years = set(pd.to_datetime(raw['timestamp']).dt.year.dropna().astype(int)) if not raw.empty and 'timestamp' in raw.columns else set()
    missing_years = [year for year in years if year not in cached_years]

    failures = []
    frames = [raw] if not raw.empty else []
    for year in missing_years:
        url = f'https://www.bangchak.co.th/en/oilprice/historical?year={year}'
        html_path = html_dir / f'bangchak_oil_historical_{year}.html'
        try:
            if html_path.exists():
                html_text = html_path.read_text(encoding='utf-8')
            else:
                with urllib.request.urlopen(url, timeout=60) as response:
                    html_text = response.read().decode('utf-8', errors='replace')
            if 'captcha' in html_text.lower() or 'perfdrive' in html_text.lower():
                raise ValueError(f'Bangchak returned anti-bot/captcha page for {year}; try fetching the HTML with curl and rerun this cell')
            parsed = parse_bangchak_historical_html(html_text)
            if parsed.empty:
                raise ValueError(f'No historical oil rows parsed for {year}')
            if not html_path.exists():
                html_path.write_text(html_text, encoding='utf-8')
            parsed['timestamp'] = pd.to_datetime(parsed['timestamp'], dayfirst=True, errors='coerce')
            parsed = parsed.dropna(subset=['timestamp'])
            frames.append(parsed)
        except Exception as exc:
            print(f'Bangchak fetch failed for {year}: {exc}')
            failures.append({'year': year, 'error': str(exc)})

    if failures:
        pd.DataFrame(failures).to_csv(fail_log_path, index=False)

    if not frames:
        return pd.DataFrame(columns=['timestamp'])

    raw = pd.concat(frames, ignore_index=True)
    raw['timestamp'] = pd.to_datetime(raw['timestamp'], errors='coerce')
    raw = raw.dropna(subset=['timestamp']).drop_duplicates('timestamp', keep='last').sort_values('timestamp')
    raw.to_csv(raw_cache_path, index=False)

    full = pd.DataFrame({'timestamp': pd.date_range(pd.Timestamp(f'{min(years)}-01-01'), pd.Timestamp(f'{max(years)}-12-31'), freq='D')})
    out = full.merge(raw, on='timestamp', how='left')
    value_cols = [c for c in out.columns if c != 'timestamp']
    # Fill only inside each calendar year that has at least one successful raw row. Missing years remain NaN.
    successful_years = set(raw['timestamp'].dt.year.astype(int))
    pieces = []
    for year, part in out.groupby(out['timestamp'].dt.year, sort=True):
        part = part.copy()
        if year in successful_years:
            part[value_cols] = part[value_cols].ffill().bfill()
        pieces.append(part)
    out = pd.concat(pieces, ignore_index=True)
    for col in value_cols:
        out[col] = pd.to_numeric(out[col], errors='coerce')
    out.to_csv(final_cache_path, index=False)
    return out


bangchak_oil = fetch_bangchak_oil_prices((2023, 2024), FETCH_BANGCHAK_OIL)
bangchak_oil.tail()


,timestamp


In [15]:
def first_existing_col(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


def combine_external_daily_features(weather_df, salary_df, ptt_df, bangchak_df):
    daily = weather_df.copy()
    daily = daily.merge(salary_df, on='timestamp', how='left')
    daily = daily.merge(ptt_df, on='timestamp', how='left')
    daily = daily.merge(bangchak_df, on='timestamp', how='left')

    # Normalize oil columns before building spreads.
    oil_cols = [c for c in daily.columns if c.startswith('ptt_') or c.startswith('bangchak_')]
    for col in oil_cols:
        daily[col] = pd.to_numeric(daily[col], errors='coerce')
    if oil_cols:
        daily[oil_cols] = daily[oil_cols].ffill().bfill()

    ptt_to_bangchak = {
        'gasohol95': ('ptt_gasohol95', ['bangchak_gasohol_95_s_evo']),
        'gasohol91': ('ptt_gasohol91', ['bangchak_gasohol_91_s_evo']),
        'gasohol_e20': ('ptt_gasohol_e20', ['bangchak_gasohol_e20_s_evo']),
        'diesel': ('ptt_diesel', ['bangchak_hi_diesel_s']),
        'premium_diesel': ('ptt_premium_diesel', ['bangchak_hi_premium_diesel_s']),
    }
    spread_cols = []
    for fuel_key, (ptt_col, bcp_candidates) in ptt_to_bangchak.items():
        bcp_col = first_existing_col(daily, bcp_candidates)
        spread_col = f'ptt_vs_bangchak_{fuel_key}_diff'
        if ptt_col in daily.columns and bcp_col:
            daily[spread_col] = daily[ptt_col] - daily[bcp_col]
        else:
            daily[spread_col] = 0.0
        spread_cols.append(spread_col)

    ptt_index_cols = [c for c in ['ptt_gasohol95', 'ptt_gasohol91', 'ptt_gasohol_e20', 'ptt_diesel'] if c in daily.columns]
    bcp_index_cols = [c for c in ['bangchak_gasohol_95_s_evo', 'bangchak_gasohol_91_s_evo', 'bangchak_gasohol_e20_s_evo', 'bangchak_hi_diesel_s'] if c in daily.columns]
    daily['ptt_fuel_price_index'] = daily[ptt_index_cols].mean(axis=1) if ptt_index_cols else 0.0
    daily['bangchak_fuel_price_index'] = daily[bcp_index_cols].mean(axis=1) if bcp_index_cols else 0.0
    daily['ptt_vs_bangchak_fuel_index_diff'] = daily['ptt_fuel_price_index'] - daily['bangchak_fuel_price_index']

    for base in ['ptt_gasohol95', 'ptt_gasohol91', 'ptt_gasohol_e20', 'ptt_diesel', 'ptt_fuel_price_index', 'bangchak_fuel_price_index']:
        if base in daily.columns:
            daily[f'{base}_change_1d'] = daily[base].diff().fillna(0)
            daily[f'{base}_ma7'] = daily[base].rolling(7, min_periods=1).mean()
            daily[f'{base}_volatility_7d'] = daily[base].rolling(7, min_periods=2).std().fillna(0)

    numeric_oil_features = [c for c in daily.columns if c.startswith('ptt_') or c.startswith('bangchak_')]
    for col in numeric_oil_features:
        daily[col] = pd.to_numeric(daily[col], errors='coerce').fillna(0)
    return daily


external_daily = combine_external_daily_features(weather, salary_calendar, ptt_oil, bangchak_oil)
external_daily.to_csv(OUTPUT_DIR / 'external_daily_features.csv', index=False)
external_daily.tail()


,timestamp,temperature_2m_mean,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum,precipitation_hours,rain_flag,heavy_rain_flag,hot_day_flag,cool_day_flag,humid_day_flag,is_month_end_salary_window,is_civil_servant_payday,is_pension_payday,is_any_salary_payday,days_since_gov_payday,days_until_gov_payday,post_payday_3d,pre_payday_3d,external_public_holiday_flag,new_year_countdown_flag,ptt_vs_bangchak_gasohol95_diff,ptt_vs_bangchak_gasohol91_diff,ptt_vs_bangchak_gasohol_e20_diff,ptt_vs_bangchak_diesel_diff,ptt_vs_bangchak_premium_diesel_diff,ptt_fuel_price_index,bangchak_fuel_price_index,ptt_vs_bangchak_fuel_index_diff,ptt_fuel_price_index_change_1d,ptt_fuel_price_index_ma7,ptt_fuel_price_index_volatility_7d,bangchak_fuel_price_index_change_1d,bangchak_fuel_price_index_ma7,bangchak_fuel_price_index_volatility_7d
726,2024-12-27,26.3,31.3,21.7,67,0.354839,0.032258,1,0,0,1,0,1,0,0,1,3,31,1,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
727,2024-12-28,26.3,31.3,21.7,67,0.354839,0.032258,1,0,0,1,0,1,0,0,1,4,31,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
728,2024-12-29,26.3,31.3,21.7,67,0.354839,0.032258,1,0,0,1,0,1,0,0,1,5,31,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
729,2024-12-30,26.3,31.3,21.7,67,0.354839,0.032258,1,0,0,1,0,1,0,0,1,6,31,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
730,2024-12-31,26.3,31.3,21.7,67,0.354839,0.032258,1,0,0,1,0,1,0,0,1,7,31,0,0,1,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 5. Build AutoGluon Training And Future Frames

Known covariates must exist for both the training history and the forecast horizon. Static features describe each item (`store_id + category`).

## 4B. Competitor Coffee Context: Inthanin And PunThai

These are market-pressure proxy features, not measured store-level competitor distances. The idea is to give the model structured context for Bangchak/Inthanin and PTG/PunThai competition, especially around fuel-station style stores where Café Amazon, Inthanin, and PunThai overlap most.


In [16]:
def build_competitor_context_features(start, end, store_df, categories):
    dates = pd.date_range(start, end, freq='D')
    base = pd.MultiIndex.from_product(
        [sorted(store_df['store_id'].unique()), categories, dates],
        names=['store_id', 'category', 'timestamp'],
    ).to_frame(index=False)
    base = base.merge(store_df[['store_id', 'neighborhood_type']], on='store_id', how='left')
    base['year'] = base['timestamp'].dt.year

    # Publicly reported / press-reported branch-count context. Values are coarse annual proxies.
    annual = pd.DataFrame([
        {
            'year': 2023,
            'inthanin_total_branches': 1000,
            'punthai_total_branches': 882,
            'punthai_gas_station_branches': 441,
            'punthai_outside_station_branches': 265,
            'punthai_franchise_branches': 176,
            'cafe_amazon_total_branches': 4200,
            'cafe_amazon_service_station_branches': 2200,
        },
        {
            'year': 2024,
            'inthanin_total_branches': 1028,
            'punthai_total_branches': 1347,
            'punthai_gas_station_branches': 696,
            'punthai_outside_station_branches': 360,
            'punthai_franchise_branches': 291,
            'cafe_amazon_total_branches': 4277,
            'cafe_amazon_service_station_branches': 2261,
        },
    ])
    base = base.merge(annual, on='year', how='left')
    branch_cols = [c for c in annual.columns if c != 'year']
    base[branch_cols] = base[branch_cols].ffill().bfill()

    inthanin_relevance = {
        'gas_station': 1.00,
        'transit': 0.45,
        'tourist': 0.45,
        'urban_residential': 0.50,
        'mall': 0.40,
        'office': 0.30,
        'university': 0.30,
        'hospital': 0.20,
    }
    punthai_relevance = {
        'gas_station': 1.00,
        'transit': 0.55,
        'tourist': 0.45,
        'urban_residential': 0.45,
        'mall': 0.30,
        'office': 0.25,
        'university': 0.25,
        'hospital': 0.15,
    }
    category_sensitivity = {
        'Coffee': 1.00,
        'Tea': 0.85,
        'Chocolate & Milk': 0.70,
        'Juice & Smoothie': 0.70,
        'Bakery': 0.55,
        'Savory Bakery': 0.45,
        'Merchandise': 0.20,
    }

    base['inthanin_store_type_relevance'] = base['neighborhood_type'].map(inthanin_relevance).fillna(0.25)
    base['punthai_store_type_relevance'] = base['neighborhood_type'].map(punthai_relevance).fillna(0.25)
    base['coffee_competitor_category_sensitivity'] = base['category'].map(category_sensitivity).fillna(0.50)

    base['inthanin_branch_index'] = base['inthanin_total_branches'] / 1000.0
    base['punthai_branch_index'] = base['punthai_total_branches'] / 1000.0
    base['punthai_gas_station_branch_index'] = base['punthai_gas_station_branches'] / 700.0
    base['cafe_amazon_station_branch_index'] = base['cafe_amazon_service_station_branches'] / 2200.0

    base['inthanin_pressure_score'] = (
        base['coffee_competitor_category_sensitivity']
        * base['inthanin_store_type_relevance']
        * base['inthanin_branch_index']
    )
    base['punthai_pressure_score'] = (
        base['coffee_competitor_category_sensitivity']
        * base['punthai_store_type_relevance']
        * base['punthai_branch_index']
    )
    base['punthai_station_pressure_score'] = (
        base['coffee_competitor_category_sensitivity']
        * base['punthai_store_type_relevance']
        * base['punthai_gas_station_branch_index']
    )
    base['coffee_chain_competition_score'] = base['inthanin_pressure_score'] + base['punthai_pressure_score']
    base['amazon_station_market_anchor_score'] = (
        base['coffee_competitor_category_sensitivity']
        * base['cafe_amazon_station_branch_index']
    )

    feature_cols = [
        'store_id', 'category', 'timestamp',
        'inthanin_total_branches', 'punthai_total_branches', 'punthai_gas_station_branches',
        'punthai_outside_station_branches', 'punthai_franchise_branches',
        'cafe_amazon_total_branches', 'cafe_amazon_service_station_branches',
        'inthanin_store_type_relevance', 'punthai_store_type_relevance', 'coffee_competitor_category_sensitivity',
        'inthanin_branch_index', 'punthai_branch_index', 'punthai_gas_station_branch_index', 'cafe_amazon_station_branch_index',
        'inthanin_pressure_score', 'punthai_pressure_score', 'punthai_station_pressure_score',
        'coffee_chain_competition_score', 'amazon_station_market_anchor_score',
    ]
    return base[feature_cols]


competitor_context_features = build_competitor_context_features(pd.Timestamp('2023-01-01'), FORECAST_END, store, CAT_ORDER)
competitor_context_features.to_csv(OUTPUT_DIR / 'competitor_context_features.csv', index=False)
competitor_context_features.head()


,store_id,category,timestamp,inthanin_total_branches,punthai_total_branches,punthai_gas_station_branches,punthai_outside_station_branches,punthai_franchise_branches,cafe_amazon_total_branches,cafe_amazon_service_station_branches,inthanin_store_type_relevance,punthai_store_type_relevance,coffee_competitor_category_sensitivity,inthanin_branch_index,punthai_branch_index,punthai_gas_station_branch_index,cafe_amazon_station_branch_index,inthanin_pressure_score,punthai_pressure_score,punthai_station_pressure_score,coffee_chain_competition_score,amazon_station_market_anchor_score
0,1,Coffee,2023-01-01,1000,882,441,265,176,4200,2200,0.3,0.25,1.0,1.0,0.882,0.63,1.0,0.3,0.2205,0.1575,0.5205,1.0
1,1,Coffee,2023-01-02,1000,882,441,265,176,4200,2200,0.3,0.25,1.0,1.0,0.882,0.63,1.0,0.3,0.2205,0.1575,0.5205,1.0
2,1,Coffee,2023-01-03,1000,882,441,265,176,4200,2200,0.3,0.25,1.0,1.0,0.882,0.63,1.0,0.3,0.2205,0.1575,0.5205,1.0
3,1,Coffee,2023-01-04,1000,882,441,265,176,4200,2200,0.3,0.25,1.0,1.0,0.882,0.63,1.0,0.3,0.2205,0.1575,0.5205,1.0
4,1,Coffee,2023-01-05,1000,882,441,265,176,4200,2200,0.3,0.25,1.0,1.0,0.882,0.63,1.0,0.3,0.2205,0.1575,0.5205,1.0


## 4C. Historical Local-Event Lift Priors

These features estimate how past event types behave by category and store type. They are computed from training history and then joined onto train/future rows as event-type priors.


In [17]:
def build_local_event_historical_lift_features_asof(target_panel_df, local_event_df, store_df, categories, cutoff):
    target_hist = target_panel_df[target_panel_df['timestamp'].le(cutoff)].rename(columns={'timestamp': 'date'}).copy()
    target_hist = target_hist.merge(store_df[['store_id', 'neighborhood_type']], on='store_id', how='left')
    target_hist['dow_num'] = pd.to_datetime(target_hist['date']).dt.dayofweek
    target_hist = target_hist.sort_values(['store_id', 'category', 'dow_num', 'date'])

    # Baseline is also as-of safe: same store/category/day-of-week, prior dates only.
    group_cols = ['store_id', 'category', 'dow_num']
    target_hist['baseline_store_category_dow_median'] = (
        target_hist.groupby(group_cols, observed=True)['units_sold']
        .transform(lambda s: s.shift(1).expanding(min_periods=3).median())
    )
    fallback = (
        target_hist.sort_values(['store_id', 'category', 'date'])
        .groupby(['store_id', 'category'], observed=True)['units_sold']
        .transform(lambda s: s.shift(1).expanding(min_periods=7).median())
    )
    target_hist['baseline_store_category_dow_median'] = target_hist['baseline_store_category_dow_median'].fillna(fallback)

    event_base = build_local_event_base(local_event_df).rename(columns={'timestamp': 'date'})
    event_base = event_base.merge(store_df[['store_id', 'neighborhood_type']], on='store_id', how='left')
    event_base['dow_num'] = pd.to_datetime(event_base['date']).dt.dayofweek

    event_target = target_hist.merge(
        event_base[['store_id', 'date', 'event_type_norm']],
        on=['store_id', 'date'],
        how='inner',
    )
    event_target['event_lift_ratio'] = (
        event_target['units_sold'] / event_target['baseline_store_category_dow_median'].replace(0, np.nan)
    )
    event_target['event_lift_ratio'] = event_target['event_lift_ratio'].replace([np.inf, -np.inf], np.nan).clip(0.25, 3.0)
    event_target = event_target.dropna(subset=['event_lift_ratio'])

    rows = []
    for event in event_base.to_dict('records'):
        event_date = pd.Timestamp(event['date'])
        prior = event_target[
            (event_target['event_type_norm'].eq(event['event_type_norm']))
            & (event_target['date'].lt(event_date))
        ]
        for category in categories:
            prior_cat = prior[prior['category'].eq(category)]
            prior_store_cat = prior_cat[prior_cat['neighborhood_type'].eq(event['neighborhood_type'])]
            tc_lift = prior_cat['event_lift_ratio'].median() if len(prior_cat) else 1.0
            tsc_lift = prior_store_cat['event_lift_ratio'].median() if len(prior_store_cat) else tc_lift
            rows.append({
                'store_id': event['store_id'],
                'category': category,
                'timestamp': event_date,
                'kaggle_event_type_category_lift_max': float(tc_lift),
                'kaggle_event_type_category_obs_max': int(len(prior_cat)),
                'kaggle_event_type_store_category_lift_max': float(tsc_lift),
                'kaggle_event_type_store_category_obs_max': int(len(prior_store_cat)),
            })
    if not rows:
        empty = pd.DataFrame(columns=['store_id', 'category', 'timestamp'])
        return empty, event_target
    raw = pd.DataFrame(rows)
    daily = (
        raw.groupby(['store_id', 'category', 'timestamp'], observed=True)
        .agg(
            kaggle_event_type_category_lift_max=('kaggle_event_type_category_lift_max', 'max'),
            kaggle_event_type_category_obs_max=('kaggle_event_type_category_obs_max', 'max'),
            kaggle_event_type_store_category_lift_max=('kaggle_event_type_store_category_lift_max', 'max'),
            kaggle_event_type_store_category_obs_max=('kaggle_event_type_store_category_obs_max', 'max'),
        )
        .reset_index()
    )
    return daily, event_target


local_event_lift_features, local_event_lift_training_audit = build_local_event_historical_lift_features_asof(
    target_panel, local_event, store, CAT_ORDER, TRAIN_END
)
local_event_lift_features.to_csv(OUTPUT_DIR / 'local_event_lift_features_asof.csv', index=False)
local_event_lift_training_audit.to_csv(OUTPUT_DIR / 'local_event_lift_training_audit.csv', index=False)
print(local_event_lift_features.shape, local_event_lift_training_audit.shape)
local_event_lift_features.head()


(9597, 7) (9100, 13)


,store_id,category,timestamp,kaggle_event_type_category_lift_max,kaggle_event_type_category_obs_max,kaggle_event_type_store_category_lift_max,kaggle_event_type_store_category_obs_max
0,1,Bakery,2023-01-05,1.000000,0,1.000000,0
1,1,Bakery,2023-02-09,0.926306,4,0.926306,0
2,1,Bakery,2023-03-14,1.302960,16,1.681034,2
3,1,Bakery,2023-03-23,1.216162,34,1.185976,8
4,1,Bakery,2023-04-05,1.282895,28,1.906061,2


In [18]:
def make_full_item_calendar(store_df, categories, horizons, start, end):
    dates = pd.date_range(start, end, freq='D')
    full = pd.MultiIndex.from_product(
        [sorted(store_df['store_id'].unique()), categories, list(horizons.keys()), dates],
        names=['store_id', 'category', 'horizon', 'timestamp'],
    ).to_frame(index=False)
    full['horizon_days'] = full['horizon'].map(horizons).astype(int)
    full['decision_date'] = full['timestamp'] - pd.to_timedelta(full['horizon_days'], unit='D')
    full['effective_decision_date'] = full['decision_date'].where(full['decision_date'].le(TRAIN_END), TRAIN_END)
    full['item_id'] = horizon_item_id(full['store_id'], full['category'], full['horizon'])
    return full


def add_date_features(df, date_dim_df):
    out = df.merge(date_dim_df.rename(columns={'date': 'timestamp'}), on='timestamp', how='left')
    out['day_of_week'] = pd.Categorical(out['day_of_week'], DOW_ORDER, ordered=True)
    out['dow_num'] = out['timestamp'].dt.dayofweek
    out['day_of_month'] = out['timestamp'].dt.day
    out['week_of_year'] = out['timestamp'].dt.isocalendar().week.astype(int)
    out['week_of_month'] = ((out['day_of_month'] - 1) // 7 + 1).astype(int)
    out['month'] = out['timestamp'].dt.month
    out['quarter'] = out['timestamp'].dt.quarter
    out['days_to_month_end'] = (out['timestamp'].dt.days_in_month - out['day_of_month']).astype(int)
    out['is_month_start'] = out['timestamp'].dt.is_month_start.astype(int)
    out['is_month_end'] = out['timestamp'].dt.is_month_end.astype(int)
    out['is_weekend'] = out['is_weekend'].fillna(out['dow_num'].ge(5)).astype(int)
    for col in ['is_holiday', 'is_school_break', 'is_payday', 'is_rainy_season']:
        out[col] = out[col].fillna(False).astype(int)
    holiday_dates = set(pd.to_datetime(date_dim_df.loc[date_dim_df['is_holiday'].fillna(False), 'date']).dt.normalize())
    timestamp_norm = out['timestamp'].dt.normalize()
    out['is_pre_holiday'] = (timestamp_norm + pd.Timedelta(days=1)).isin(holiday_dates).astype(int)
    out['is_post_holiday'] = (timestamp_norm - pd.Timedelta(days=1)).isin(holiday_dates).astype(int)
    out['is_long_weekend_adjacent'] = ((out['is_pre_holiday'].eq(1) | out['is_post_holiday'].eq(1)) & out['is_weekend'].eq(1)).astype(int)
    out['sin_doy'] = np.sin(2 * np.pi * out['timestamp'].dt.dayofyear / 365.25)
    out['cos_doy'] = np.cos(2 * np.pi * out['timestamp'].dt.dayofyear / 365.25)
    return out


def add_effective_decision_features(df, date_dim_df):
    decision_cal = date_dim_df.rename(columns={
        'date': 'effective_decision_date',
        'is_weekend': 'decision_is_weekend',
        'is_holiday': 'decision_is_holiday',
        'is_payday': 'decision_is_payday',
        'is_school_break': 'decision_is_school_break',
        'is_rainy_season': 'decision_is_rainy_season',
    })[['effective_decision_date', 'decision_is_weekend', 'decision_is_holiday', 'decision_is_payday', 'decision_is_school_break', 'decision_is_rainy_season']]
    out = df.merge(decision_cal, on='effective_decision_date', how='left')
    out['decision_dow_num'] = out['effective_decision_date'].dt.dayofweek
    out['decision_day_of_month'] = out['effective_decision_date'].dt.day
    out['decision_month'] = out['effective_decision_date'].dt.month
    out['decision_sin_doy'] = np.sin(2 * np.pi * out['effective_decision_date'].dt.dayofyear / 365.25)
    out['decision_cos_doy'] = np.cos(2 * np.pi * out['effective_decision_date'].dt.dayofyear / 365.25)
    out['days_from_decision_to_forecast'] = (out['timestamp'] - out['effective_decision_date']).dt.days.clip(lower=0)
    for col in ['decision_is_weekend', 'decision_is_holiday', 'decision_is_payday', 'decision_is_school_break', 'decision_is_rainy_season']:
        out[col] = out[col].fillna(False).astype(int)
    return out


def build_model_frame():
    full = make_full_item_calendar(store, CAT_ORDER, HORIZON_TO_DAYS, pd.Timestamp('2023-01-01'), FORECAST_END)
    target = target_panel[['store_id', 'category', 'timestamp', 'units_sold']]
    frame = full.merge(target, on=['store_id', 'category', 'timestamp'], how='left')
    frame = frame.merge(store, on='store_id', how='left')
    frame = add_date_features(frame, date_dim)
    frame = add_effective_decision_features(frame, date_dim)
    frame = frame.merge(promo_features, on=['store_id', 'category', 'timestamp'], how='left')
    frame = frame.merge(local_event_features, on=['store_id', 'timestamp'], how='left')
    frame = frame.merge(local_event_category_features, on=['store_id', 'category', 'timestamp'], how='left')
    frame = frame.merge(local_event_lift_features, on=['store_id', 'category', 'timestamp'], how='left')
    frame = frame.merge(special_features, on=['store_id', 'category', 'timestamp'], how='left')
    frame = frame.merge(external_daily, on='timestamp', how='left')
    frame = frame.merge(competitor_context_features, on=['store_id', 'category', 'timestamp'], how='left')
    frame = frame.merge(subcategory_cluster_features, on='category', how='left')
    frame = frame.merge(serve_type_weather_features, on='category', how='left')
    frame = frame.merge(stockout_asof, on=['store_id', 'category', 'effective_decision_date'], how='left')

    fill_zero_cols = [
        'active_promo_products', 'max_discount_pct', 'mean_discount_pct', 'email_sent', 'social_campaign', 'promo_type_count',
        'promo_channel_count', 'promo_discount_intensity', 'has_deep_discount',
        'kaggle_local_event_count', 'kaggle_local_event_type_count', 'kaggle_has_music_event', 'kaggle_has_market_event', 'kaggle_has_cultural_event',
        'kaggle_event_intensity_sum', 'kaggle_event_intensity_max', 'kaggle_event_name_length_mean',
        'kaggle_local_event_count_lag1', 'kaggle_local_event_count_lead1', 'kaggle_local_event_window_3d',
        'kaggle_days_since_local_event', 'kaggle_days_until_local_event',
        'kaggle_event_category_match_count', 'kaggle_event_store_relevance_max', 'kaggle_event_category_relevance_max', 'kaggle_event_relevance_score',
        'kaggle_event_type_category_lift_max', 'kaggle_event_type_category_obs_max',
        'kaggle_event_type_store_category_lift_max', 'kaggle_event_type_store_category_obs_max',
        'special_event_count', 'special_event_category_match', 'special_event_store_weight', 'special_event_score',
        *STOCKOUT_ASOF_FEATURES,
        'inthanin_total_branches', 'punthai_total_branches', 'punthai_gas_station_branches',
        'punthai_outside_station_branches', 'punthai_franchise_branches',
        'cafe_amazon_total_branches', 'cafe_amazon_service_station_branches',
        'inthanin_store_type_relevance', 'punthai_store_type_relevance', 'coffee_competitor_category_sensitivity',
        'inthanin_branch_index', 'punthai_branch_index', 'punthai_gas_station_branch_index', 'cafe_amazon_station_branch_index',
        'inthanin_pressure_score', 'punthai_pressure_score', 'punthai_station_pressure_score',
        'coffee_chain_competition_score', 'amazon_station_market_anchor_score',
        'subcategory_cluster_count', 'subcategory_cluster_entropy', 'subcategory_cluster_dominant_share',
    ]
    subcategory_numeric_cols = [
        c for c in frame.columns
        if c.startswith('subcategory_cluster_') and c != 'subcategory_cluster_dominant'
    ]
    serve_type_numeric_cols = [c for c in frame.columns if c.startswith('serve_type_')]
    fill_zero_cols = fill_zero_cols + subcategory_numeric_cols + serve_type_numeric_cols
    for col in fill_zero_cols:
        if col in frame.columns:
            frame[col] = frame[col].fillna(0)

    if 'subcategory_cluster_dominant' in frame.columns:
        frame['subcategory_cluster_dominant'] = frame['subcategory_cluster_dominant'].fillna('0').astype(str).astype('category')

    if 'serve_type_temp_response_score' in frame.columns:
        for temp_col, suffix in [('temperature_2m_mean', 'mean'), ('temperature_2m_min', 'min')]:
            if temp_col in frame.columns:
                frame[f'serve_type_temp_response_x_temperature_{suffix}'] = (
                    frame['serve_type_temp_response_score'] * pd.to_numeric(frame[temp_col], errors='coerce').fillna(0)
                )
                frame[f'serve_type_positive_share_x_temperature_{suffix}'] = (
                    frame['serve_type_temp_positive_share'] * pd.to_numeric(frame[temp_col], errors='coerce').fillna(0)
                )
                frame[f'serve_type_negative_share_x_temperature_{suffix}'] = (
                    frame['serve_type_temp_negative_share'] * pd.to_numeric(frame[temp_col], errors='coerce').fillna(0)
                )

    dynamic_event_cols = [
        col for col in frame.columns
        if col.startswith('kaggle_event_type_') or col.startswith('kaggle_event_kw_')
    ]
    if dynamic_event_cols:
        frame[dynamic_event_cols] = frame[dynamic_event_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

    bool_cols = ['has_drive_through']
    for col in bool_cols:
        if col in frame.columns:
            frame[col] = frame[col].fillna(False).astype(int)

    frame['category'] = pd.Categorical(frame['category'], CAT_ORDER, ordered=True)
    frame['horizon'] = pd.Categorical(frame['horizon'], list(HORIZON_TO_DAYS.keys()), ordered=True)
    frame['neighborhood_type'] = frame['neighborhood_type'].astype('category')
    frame['store_id_cat'] = frame['store_id'].astype(str).astype('category')
    frame['open_hour'] = pd.to_numeric(frame['open_time'].str[:2], errors='coerce')
    frame['close_hour'] = pd.to_numeric(frame['close_time'].str[:2], errors='coerce')
    frame['operating_hours'] = frame['close_hour'] - frame['open_hour']
    frame['days_since_open'] = (frame['timestamp'] - frame['opened_date']).dt.days.clip(lower=0)
    frame['store_age_months'] = (frame['days_since_open'] / 30.4375).clip(lower=0)
    frame['is_new_store_90d'] = frame['days_since_open'].le(90).astype(int)
    return frame.copy()


model_frame = build_model_frame()
print(model_frame.shape)
model_frame.head()


/var/folders/zm/l71z21jd7gg4w4y6px1y79k80000gn/T/ipykernel_8187/3981413027.py:56: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out[col] = out[col].fillna(False).astype(int)
/var/folders/zm/l71z21jd7gg4w4y6px1y79k80000gn/T/ipykernel_8187/3981413027.py:56: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out[col] = out[col].fillna(False).astype(int)
/var/folders/zm/l71z21jd7gg4w4y6px1y79k80000gn/T/ipykernel_8187/3981413027.py:56: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call 

(307020, 210)


,store_id,category,horizon,timestamp,horizon_days,decision_date,effective_decision_date,item_id,units_sold,neighborhood_type,seating_capacity,has_drive_through,staff_count,open_time,close_time,opened_date,day_of_week,week_number,month,quarter,year,is_weekend,is_holiday,holiday_name,is_school_break,is_payday,is_rainy_season,dow_num,day_of_month,week_of_year,week_of_month,days_to_month_end,is_month_start,is_month_end,is_pre_holiday,is_post_holiday,is_long_weekend_adjacent,sin_doy,cos_doy,decision_is_weekend,decision_is_holiday,decision_is_payday,decision_is_school_break,decision_is_rainy_season,decision_dow_num,decision_day_of_month,decision_month,decision_sin_doy,decision_cos_doy,days_from_decision_to_forecast,active_promo_products,max_discount_pct,mean_discount_pct,email_sent,social_campaign,promo_type_count,promo_channel_count,promo_discount_intensity,has_deep_discount,kaggle_local_event_count,...,ptt_fuel_price_index_ma7,ptt_fuel_price_index_volatility_7d,bangchak_fuel_price_index_change_1d,bangchak_fuel_price_index_ma7,bangchak_fuel_price_index_volatility_7d,inthanin_total_branches,punthai_total_branches,punthai_gas_station_branches,punthai_outside_station_branches,punthai_franchise_branches,cafe_amazon_total_branches,cafe_amazon_service_station_branches,inthanin_store_type_relevance,punthai_store_type_relevance,coffee_competitor_category_sensitivity,inthanin_branch_index,punthai_branch_index,punthai_gas_station_branch_index,cafe_amazon_station_branch_index,inthanin_pressure_score,punthai_pressure_score,punthai_station_pressure_score,coffee_chain_competition_score,amazon_station_market_anchor_score,serve_type_blended_product_share,serve_type_hot_product_share,serve_type_iced_product_share,serve_type_piece_product_share,serve_type_blended_product_count,serve_type_hot_product_count,serve_type_iced_product_count,serve_type_piece_product_count,serve_type_count,serve_type_temp_response_score,serve_type_temp_positive_share,serve_type_temp_negative_share,serve_type_piece_share,hist_stockout_rate,hist_stockout_sku_share,hist_mean_closing_stock,stockout_rate_7d,stockout_rate_28d,stockout_sku_share_7d,stockout_sku_share_28d,closing_stock_mean_7d,closing_stock_mean_28d,days_since_stockout,serve_type_temp_response_x_temperature_mean,serve_type_positive_share_x_temperature_mean,serve_type_negative_share_x_temperature_mean,serve_type_temp_response_x_temperature_min,serve_type_positive_share_x_temperature_min,serve_type_negative_share_x_temperature_min,store_id_cat,open_hour,close_hour,operating_hours,days_since_open,store_age_months,is_new_store_90d
0,1,Coffee,1d,2023-01-01,1,2022-12-31,2022-12-31,1_Coffee_1d,100.0,university,39,0,4,06:00,21:00,2019-12-21,Sunday,52,1,1,2023,1,1,วันขึ้นปีใหม่,0,0,0,6,1,52,1,30,1,0,0,0,0,0.017202,0.999852,0,0,0,0,0,5,31,12,-0.004301,0.999991,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1000,882,441,265,176,4200,2200,0.3,0.25,1.0,1.0,0.882,0.63,1.0,0.3,0.2205,0.1575,0.5205,1.0,0.176471,0.411765,0.411765,0.0,3.0,7.0,7.0,0.0,3,0.108882,0.588235,0.411765,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,2.907159,15.705882,10.994118,2.373635,12.823529,8.976471,1,6,21,15,1107,36.369610,0
1,1,Coffee,1d,2023-01-02,1,2023-01-01,2023-01-01,1_Coffee_1d,140.0,university,39,0,4,06:00,21:00,2019-12-21,Monday,1,1,1,2023,0,0,NaN,0,0,0,0,2,1,1,29,0,0,0,1,0,0.034398,0.999408,1,1,0,0,0,6,1,1,0.017202,0.999852,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1000,882,441,265,176,4200,2200,0.3,0.25,1.0,1.0,0.882,0.63,1.0,0.3,0.2205,0.1575,0.5205,1.0,0.176471,0.411765,0.411765,0.0,3.0,7.0,7.0,0.0,3,0.108882,0.588235,0.411765,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,99.0,2.907159,15.705882,10.994118,2.373635,12.823529,8.976471,1,6,21,15,1108,36.402464,0
2,1,Coffee,1d,2023-01-03,1,2023-01-02,2023-01-02,1_Coffee_1d,109.0,university,39,0,4,06:00,21:00,2019-12-21,Tuesday,1,1,1,2023,0,0,NaN,0,0,0,1,3,1,1,28,0,0,0,0,0,0.051584,0.998669,0,0,0,0,0

In [19]:
known_covariates = [
    'day_of_week', 'dow_num', 'day_of_month', 'week_of_month', 'days_to_month_end', 'month', 'quarter',
    'is_weekend', 'is_holiday', 'is_pre_holiday', 'is_post_holiday', 'is_long_weekend_adjacent',
    'is_school_break', 'is_payday', 'is_rainy_season', 'is_month_start', 'is_month_end',
    'sin_doy', 'cos_doy',
    'decision_dow_num', 'decision_day_of_month', 'decision_month',
    'decision_is_weekend', 'decision_is_holiday', 'decision_is_payday', 'decision_is_school_break', 'decision_is_rainy_season',
    'decision_sin_doy', 'decision_cos_doy', 'days_from_decision_to_forecast',
    'active_promo_products', 'max_discount_pct', 'mean_discount_pct', 'email_sent', 'social_campaign', 'promo_type_count',
    'promo_channel_count', 'promo_discount_intensity', 'has_deep_discount',
    'kaggle_local_event_type_count', 'kaggle_has_music_event', 'kaggle_has_cultural_event',
    'kaggle_event_intensity_sum', 'kaggle_event_intensity_max',
    'kaggle_local_event_count_lead1', 'kaggle_days_until_local_event',
    'kaggle_event_store_relevance_max', 'kaggle_event_relevance_score',
    'temperature_2m_mean', 'temperature_2m_min',
    'is_any_salary_payday',
    'serve_type_temp_response_x_temperature_mean', 'serve_type_positive_share_x_temperature_mean', 'serve_type_negative_share_x_temperature_mean',
    'serve_type_temp_response_x_temperature_min', 'serve_type_positive_share_x_temperature_min', 'serve_type_negative_share_x_temperature_min',
    *STOCKOUT_ASOF_FEATURES,
    'days_since_open', 'store_age_months', 'is_new_store_90d',
    'punthai_outside_station_branches', 'cafe_amazon_station_branch_index',
    'inthanin_pressure_score', 'punthai_station_pressure_score', 'amazon_station_market_anchor_score',
]
known_covariates += [
    c for c in model_frame.columns
    if (c.startswith('kaggle_event_type_') or c.startswith('kaggle_event_kw_')) and c not in known_covariates
]

LEAKAGE_SAFE_PRUNE_BLOCKLIST = {
    # Weak or raw event fields from v5 pruning
    'kaggle_event_name_length_mean', 'kaggle_local_event_count', 'kaggle_has_market_event',
    'kaggle_event_kw_food_category_score', 'kaggle_event_kw_sport_category_score',
    'kaggle_event_kw_market_category_score', 'kaggle_event_kw_night',
    'kaggle_event_kw_music', 'kaggle_event_kw_sport',
    # Target-derived event lift priors are unsafe for row-specific horizon cutoffs.
    'kaggle_event_type_category_lift_max', 'kaggle_event_type_category_obs_max',
    'kaggle_event_type_store_category_lift_max', 'kaggle_event_type_store_category_obs_max',
    'kaggle_days_since_local_event',
    'kaggle_event_type_convention_category_score', 'kaggle_event_type_concert_category_score',
    'kaggle_event_category_match_count', 'kaggle_event_category_relevance_max',
    'kaggle_local_event_window_3d', 'kaggle_local_event_count_lag1',
    'kaggle_event_type_convention', 'kaggle_event_type_music_festival',
    'kaggle_event_type_sports', 'kaggle_event_kw_coffee', 'kaggle_event_kw_food',
    'kaggle_event_kw_student', 'kaggle_event_kw_auto', 'kaggle_event_kw_religious',
    'kaggle_event_type_market', 'kaggle_event_type_cultural', 'kaggle_event_kw_book', 'kaggle_event_kw_market',
    # Synthetic/special event fields that did not add validation signal in v5
    'special_event_count', 'special_event_category_match', 'special_event_store_weight',
    'special_event_score', 'new_year_countdown_flag',
    # Oil fields are excluded from this strict notebook because future prices are unknown at cutoff
}
known_covariates_before_prune = [c for c in known_covariates if c in model_frame.columns]
known_covariates = [
    c for c in known_covariates_before_prune
    if c not in LEAKAGE_SAFE_PRUNE_BLOCKLIST
    and not c.startswith(('ptt_', 'bangchak_'))
    and 'ptt_vs_bangchak' not in c
]
pruned_known_covariates = sorted(set(known_covariates_before_prune) - set(known_covariates))

subcategory_static_cols = [c for c in model_frame.columns if c.startswith('subcategory_cluster_')]
serve_type_static_cols = [
    c for c in model_frame.columns
    if c.startswith('serve_type_') and '_x_temperature_' not in c
]


def build_autogluon_artifacts(horizons, artifact_name):
    subset = model_frame[model_frame['horizon'].astype(str).isin(horizons)].copy()
    if subset.empty:
        raise ValueError(f'No rows found for {artifact_name} horizons: {horizons}')

    static = (
        subset.groupby('item_id', observed=True)
        .agg(
            store_id=('store_id', 'first'),
            category=('category', 'first'),
            horizon=('horizon', 'first'),
            horizon_days=('horizon_days', 'first'),
            neighborhood_type=('neighborhood_type', 'first'),
            has_drive_through=('has_drive_through', 'first'),
            staff_count=('staff_count', 'first'),
            open_hour=('open_hour', 'first'),
            close_hour=('close_hour', 'first'),
            operating_hours=('operating_hours', 'first'),
            is_new_store_at_train_end=('is_new_store_90d', 'last'),
            **{col: (col, 'first') for col in subcategory_static_cols + serve_type_static_cols},
        )
    )
    static['store_id'] = static['store_id'].astype(str).astype('category')
    static['category'] = static['category'].astype('category')
    static['horizon'] = static['horizon'].astype('category')
    static['neighborhood_type'] = static['neighborhood_type'].astype('category')
    if 'subcategory_cluster_dominant' in static.columns:
        static['subcategory_cluster_dominant'] = static['subcategory_cluster_dominant'].astype(str).astype('category')

    train_part = subset[subset['timestamp'].le(TRAIN_END) & subset['units_sold'].notna()].copy()
    future_part = subset[subset['timestamp'].between(FORECAST_START, FORECAST_END)].copy()
    train_columns = ['item_id', 'timestamp', 'units_sold'] + known_covariates
    future_columns = ['item_id', 'timestamp'] + known_covariates

    train_ts = TimeSeriesDataFrame.from_data_frame(
        train_part[train_columns],
        id_column='item_id',
        timestamp_column='timestamp',
    )
    train_ts.static_features = static

    future_ts = TimeSeriesDataFrame.from_data_frame(
        future_part[future_columns],
        id_column='item_id',
        timestamp_column='timestamp',
    )

    print(f'{artifact_name} horizons:', horizons)
    print(f'{artifact_name} train/future/static:', train_ts.shape, future_ts.shape, static.shape)
    return {
        'name': artifact_name,
        'horizons': horizons,
        'model_frame': subset,
        'train_df': train_part,
        'future_cov_df': future_part,
        'train_cols': train_columns,
        'future_cols': future_columns,
        'train_tsdf': train_ts,
        'future_known_covariates': future_ts,
        'static_features': static,
    }


short_artifacts = build_autogluon_artifacts(SHORT_HORIZONS, 'short_1d_7d')
long_artifacts = build_autogluon_artifacts(LONG_HORIZONS, 'long_1m')
ARTIFACTS_BY_ROUTE = {
    'short': short_artifacts,
    'long': long_artifacts,
}

# Compatibility aliases for downstream sanity checks / optional feature importance.
train_df = pd.concat([short_artifacts['train_df'], long_artifacts['train_df']], ignore_index=True)
future_cov_df = pd.concat([short_artifacts['future_cov_df'], long_artifacts['future_cov_df']], ignore_index=True)
train_cols = short_artifacts['train_cols']
future_cols = short_artifacts['future_cols']
train_tsdf = short_artifacts['train_tsdf']
future_known_covariates = short_artifacts['future_known_covariates']
static_features = pd.concat([short_artifacts['static_features'], long_artifacts['static_features']])

print('Known covariates:', len(known_covariates))
print('Subcategory static features:', subcategory_static_cols)
print('Serve-type static features:', serve_type_static_cols)
print('Pruned known covariates:', len(pruned_known_covariates))
print(pruned_known_covariates)
print(known_covariates)


Known covariates: 91
Subcategory static features: []
Serve-type static features: ['serve_type_blended_product_share', 'serve_type_hot_product_share', 'serve_type_iced_product_share', 'serve_type_piece_product_share', 'serve_type_blended_product_count', 'serve_type_hot_product_count', 'serve_type_iced_product_count', 'serve_type_piece_product_count', 'serve_type_count', 'serve_type_temp_response_score', 'serve_type_temp_positive_share', 'serve_type_temp_negative_share', 'serve_type_piece_share']
Pruned known covariates: 24
['kaggle_event_kw_auto', 'kaggle_event_kw_book', 'kaggle_event_kw_coffee', 'kaggle_event_kw_food', 'kaggle_event_kw_food_category_score', 'kaggle_event_kw_market', 'kaggle_event_kw_market_category_score', 'kaggle_event_kw_music', 'kaggle_event_kw_night', 'kaggle_event_kw_religious', 'kaggle_event_kw_sport', 'kaggle_event_kw_sport_category_score', 'kaggle_event_kw_student', 'kaggle_event_type_category_lift_max', 'kaggle_event_type_category_obs_max', 'kaggle_event_type_

## 6. Fit AutoGluon TimeSeriesPredictor

This mirrors the reference notebook’s AutoGluon step, with known covariates and item static features added.

In [20]:
def fit_or_load_autogluon_predictor(artifacts, model_path, hyperparameters, label):
    if RUN_TRAINING:
        fit_kwargs = dict(
            train_data=artifacts['train_tsdf'],
            time_limit=TIME_LIMIT_SECONDS,
            hyperparameters=hyperparameters,
            num_val_windows=2,
            refit_every_n_windows=1,
        )
        if PRESETS is not None:
            fit_kwargs['presets'] = PRESETS

        predictor = TimeSeriesPredictor(
            target='units_sold',
            prediction_length=PREDICTION_LENGTH,
            freq='D',
            eval_metric='MAE',
            known_covariates_names=known_covariates,
            path=str(model_path),
        ).fit(**fit_kwargs)
        print(f'Leaderboard: {label}')
        display(predictor.leaderboard())
        return predictor

    predictor = TimeSeriesPredictor.load(str(model_path))
    print(f'Loaded existing predictor: {label} -> {model_path}')
    return predictor


short_predictor = fit_or_load_autogluon_predictor(
    short_artifacts,
    SHORT_MODEL_PATH,
    SHORT_HORIZON_HYPERPARAMETERS,
    'short horizons 1d+7d',
)
long_predictor = fit_or_load_autogluon_predictor(
    long_artifacts,
    LONG_MODEL_PATH,
    LONG_HORIZON_HYPERPARAMETERS,
    'long horizon 1m',
)
PREDICTORS_BY_ROUTE = {
    'short': short_predictor,
    'long': long_predictor,
}


Beginning AutoGluon training... Time limit = 3600s
AutoGluon will save models to '/Users/moi/PROJECT/Expresso/outputs/autogluon_weighted_leakage_safe_horizon/model_weighted_chronos_direct_strict_horizon_safe_no_target_lift'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.4
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.3.0: Thu Jan  2 20:24:06 PST 2025; root:xnu-11215.81.4~3/RELEASE_ARM64_T8103
CPU Count:          4
Pytorch Version:    2.9.1
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       1.40 GB / 8.00 GB (17.5%)
Disk Space Avail:   13.66 GB / 228.27 GB (6.0%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': MAE,
 'freq': 'D',
 'hyperparameters': {'Chronos': {'ag_args': {'name_suffix': 'WithRegressor'},
                                 'covariate_regressor'

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/191M [00:00<?, ?B/s]

	To avoid this warning, specify the model hyperparameter "ag.max_memory_usage_ratio" to a larger value (currently 1.0, set to >=1.06 to avoid the warning)
		To set the same value for all models, do the following when calling predictor.fit: `predictor.fit(..., ag_args_fit={"ag.max_memory_usage_ratio": VALUE})`
		Setting "ag.max_memory_usage_ratio" to values above 1 may result in out-of-memory errors. You may consider using a machine with more memory as a safer alternative.
	-7.7861       = Validation score (-MAE)
	229.94  s     = Training runtime
	4.70    s     = Validation (prediction) runtime
Fitting 1 ensemble(s), in 1 layers.
Training ensemble model WeightedEnsemble. Training for up to 3305.6s.
	Ensemble weights: {'ChronosWithRegressor[bolt_small]': 0.77, 'DirectTabular': 0.23}
	-7.7427       = Validation score (-MAE)
	0.98    s     = Training runtime
	7.74    s     = Validation (prediction) runtime
Training complete. Models trained: ['RecursiveTabular', 'DirectTabular', 'ChronosWit

                              model  score_val  pred_time_val  fit_time_marginal  fit_order
0                  WeightedEnsemble  -7.742664       7.738041           0.978879          4
1  ChronosWithRegressor[bolt_small]  -7.786120       4.700649         229.935480          3
2                     DirectTabular  -8.164384       3.033739          17.223861          2
3                  RecursiveTabular -10.267775       2.090509          23.461534          1


## 6B. Why 1m Is Not Defaulting To TimesFM Yet

TimesFM is a strong candidate for the `1m` horizon because it is a pretrained time-series foundation model and newer releases support longer context plus covariates/XReg. For this competition notebook, the safer production default remains AutoGluon for `1m` because the model can directly use the existing leakage-safe covariates and static features. A clean next experiment is to create a TimesFM-only `1m` benchmark from `long_artifacts`, compare it on the same cutoff backtest, then blend it with `long_predictor` only if it improves validation.


## 7. Strict Horizon-Safe Predict And Export Submission

This section predicts in batches by `effective_decision_date`. Each batch passes AutoGluon only the target history available at that cutoff, then selects the requested forecast date from the 61-day prediction window.

This fixes the earlier horizon leak where `predict(train_tsdf, ...)` gave `7d` and `1m` rows target history through `2024-10-31` even when their decision date was earlier.


In [21]:
def make_tsdf_for_cutoff(cutoff, artifacts, item_ids=None):
    cutoff = pd.Timestamp(cutoff)
    history = artifacts['train_df'][artifacts['train_df']['timestamp'].le(cutoff)].copy()
    if item_ids is not None:
        history = history[history['item_id'].isin(item_ids)].copy()
    if history.empty:
        raise ValueError(f"No target history available through cutoff {cutoff.date()} for {artifacts['name']}")
    tsdf = TimeSeriesDataFrame.from_data_frame(
        history[artifacts['train_cols']],
        id_column='item_id',
        timestamp_column='timestamp',
    )
    tsdf.static_features = artifacts['static_features'].loc[
        artifacts['static_features'].index.intersection(history['item_id'].unique())
    ]
    return tsdf


def apply_batch_cutoff_covariates(future, cutoff):
    cutoff = pd.Timestamp(cutoff)
    future = future.copy()

    cutoff_row = date_dim[date_dim['date'].eq(cutoff)]
    if cutoff_row.empty:
        raise ValueError(f'No DATE_DIM row for cutoff {cutoff.date()}')
    cutoff_row = cutoff_row.iloc[0]
    future['decision_dow_num'] = cutoff.dayofweek
    future['decision_day_of_month'] = cutoff.day
    future['decision_month'] = cutoff.month
    future['decision_is_weekend'] = int(bool(cutoff_row.get('is_weekend', cutoff.dayofweek >= 5)))
    future['decision_is_holiday'] = int(bool(cutoff_row.get('is_holiday', False)))
    future['decision_is_payday'] = int(bool(cutoff_row.get('is_payday', False)))
    future['decision_is_school_break'] = int(bool(cutoff_row.get('is_school_break', False)))
    future['decision_is_rainy_season'] = int(bool(cutoff_row.get('is_rainy_season', False)))
    future['decision_sin_doy'] = np.sin(2 * np.pi * cutoff.dayofyear / 365.25)
    future['decision_cos_doy'] = np.cos(2 * np.pi * cutoff.dayofyear / 365.25)
    future['days_from_decision_to_forecast'] = (future['timestamp'] - cutoff).dt.days.clip(lower=0)

    stock_cols = STOCKOUT_ASOF_FEATURES
    stock_cutoff = stockout_asof[stockout_asof['effective_decision_date'].eq(cutoff)][['store_id', 'category'] + stock_cols]
    future = future.drop(columns=[c for c in stock_cols if c in future.columns]).merge(
        stock_cutoff,
        on=['store_id', 'category'],
        how='left',
    )
    for col in stock_cols:
        future[col] = pd.to_numeric(future[col], errors='coerce').fillna(0)
    return future


def make_future_covariates_for_cutoff(cutoff, artifacts, item_ids=None):
    cutoff = pd.Timestamp(cutoff)
    horizon_end = cutoff + pd.Timedelta(days=PREDICTION_LENGTH)
    future = artifacts['model_frame'][
        artifacts['model_frame']['timestamp'].between(cutoff + pd.Timedelta(days=1), horizon_end)
    ].copy()
    if item_ids is not None:
        future = future[future['item_id'].isin(item_ids)].copy()
    if future.empty:
        raise ValueError(f"No known covariates available after cutoff {cutoff.date()} for {artifacts['name']}")
    future = apply_batch_cutoff_covariates(future, cutoff)
    return TimeSeriesDataFrame.from_data_frame(
        future[artifacts['future_cols']],
        id_column='item_id',
        timestamp_column='timestamp',
    )


def predict_horizon_safe_submission(predictors_by_route, artifacts_by_route, sample_df):
    pieces = []
    audit_rows = []
    batch_rows = []
    sample_work = sample_df.copy()
    sample_work['effective_decision_date'] = pd.to_datetime(sample_work['effective_decision_date'])
    sample_work['forecast_date'] = pd.to_datetime(sample_work['forecast_date'])
    sample_work['model_route'] = sample_work['horizon'].map(HORIZON_MODEL_ROUTE)
    if sample_work['model_route'].isna().any():
        bad = sorted(sample_work.loc[sample_work['model_route'].isna(), 'horizon'].unique())
        raise ValueError(f'No model route configured for horizons: {bad}')

    for cutoff, cutoff_group in sample_work.groupby('effective_decision_date', sort=True):
        cutoff = pd.Timestamp(cutoff)
        for route, group in cutoff_group.groupby('model_route', sort=True):
            artifacts = artifacts_by_route[route]
            predictor = predictors_by_route[route]
            item_ids = sorted(group['item_id'].unique())
            max_requested_offset = int((group['forecast_date'].max() - cutoff).days)
            if max_requested_offset > PREDICTION_LENGTH:
                raise ValueError(
                    f'Cutoff {cutoff.date()} route={route} needs {max_requested_offset} days, '
                    f'but predictor prediction_length is {PREDICTION_LENGTH}'
                )

            context_tsdf = make_tsdf_for_cutoff(cutoff, artifacts, item_ids=item_ids)
            future_covariates = make_future_covariates_for_cutoff(cutoff, artifacts, item_ids=item_ids)
            preds = predictor.predict(context_tsdf, known_covariates=future_covariates)
            pred_mean = (
                preds.reset_index()[['item_id', 'timestamp', 'mean']]
                .rename(columns={'timestamp': 'forecast_date', 'mean': 'units_sold_predicted'})
            )

            merged = group.merge(pred_mean, on=['item_id', 'forecast_date'], how='left')
            missing = int(merged['units_sold_predicted'].isna().sum())
            if missing:
                raise ValueError(f'Missing {missing} predictions for cutoff {cutoff.date()} route={route}')
            merged['units_sold_predicted'] = merged['units_sold_predicted'].clip(lower=0)
            pieces.append(merged)

            context_max = context_tsdf.reset_index()['timestamp'].max()
            context_min = context_tsdf.reset_index()['timestamp'].min()
            batch_rows.append({
                'model_route': route,
                'horizons': ','.join(sorted(group['horizon'].unique())),
                'effective_decision_date': cutoff,
                'context_min_timestamp': context_min,
                'context_max_timestamp': context_max,
                'rows_predicted': len(group),
                'items_predicted': len(item_ids),
                'max_requested_offset_days': max_requested_offset,
                'prediction_length': PREDICTION_LENGTH,
            })
            audit_part = merged[['id', 'store_id', 'category', 'horizon', 'horizon_days', 'forecast_date', 'decision_date', 'effective_decision_date', 'model_route']].copy()
            audit_part['context_max_timestamp'] = context_max
            audit_part['future_dynamic_covariate_cutoff'] = cutoff
            audit_part['forecast_offset_days'] = (audit_part['forecast_date'] - audit_part['context_max_timestamp']).dt.days
            audit_part['history_after_effective_decision'] = audit_part['context_max_timestamp'].gt(audit_part['effective_decision_date'])
            audit_part['dynamic_covariate_after_context'] = audit_part['future_dynamic_covariate_cutoff'].gt(audit_part['context_max_timestamp'])
            audit_part['forecast_outside_prediction_window'] = audit_part['forecast_offset_days'].gt(PREDICTION_LENGTH)
            audit_rows.append(audit_part)
            print(
                f'route={route:5s} cutoff={cutoff.date()} rows={len(group):4d} items={len(item_ids):3d} '
                f'context_max={pd.Timestamp(context_max).date()} max_offset={max_requested_offset}'
            )

    out = pd.concat(pieces, ignore_index=True).sort_values('id')
    audit = pd.concat(audit_rows, ignore_index=True).sort_values('id')
    batches = pd.DataFrame(batch_rows).sort_values(['effective_decision_date', 'model_route'])

    leak_count = int(audit['history_after_effective_decision'].sum())
    dynamic_covariate_leak_count = int(audit['dynamic_covariate_after_context'].sum())
    outside_count = int(audit['forecast_outside_prediction_window'].sum())
    if leak_count or dynamic_covariate_leak_count or outside_count:
        raise AssertionError(
            f'Horizon safety audit failed: history leaks={leak_count}, dynamic covariate leaks={dynamic_covariate_leak_count}, outside prediction window={outside_count}'
        )
    return out, audit, batches


if 'PREDICTORS_BY_ROUTE' not in globals():
    short_predictor = TimeSeriesPredictor.load(str(SHORT_MODEL_PATH))
    long_predictor = TimeSeriesPredictor.load(str(LONG_MODEL_PATH))
    PREDICTORS_BY_ROUTE = {'short': short_predictor, 'long': long_predictor}
    print('Loaded existing split-horizon predictors')

strict_submission, horizon_audit, horizon_batches = predict_horizon_safe_submission(
    PREDICTORS_BY_ROUTE,
    ARTIFACTS_BY_ROUTE,
    sample_parsed,
)

final_submission = strict_submission[['id', 'units_sold_predicted']].copy()
final_submission['units_sold_predicted'] = final_submission['units_sold_predicted'].clip(lower=0)
submission_path = OUTPUT_DIR / 'submission_split_horizon_short_ag_long_ag.csv'
legacy_submission_path = OUTPUT_DIR / 'submission_weighted_leakage_safe_horizon.csv'
audit_path = OUTPUT_DIR / 'horizon_safe_prediction_audit_split_horizon.csv'
batch_path = OUTPUT_DIR / 'horizon_safe_prediction_batches_split_horizon.csv'

final_submission.to_csv(submission_path, index=False)
final_submission.to_csv(legacy_submission_path, index=False)
horizon_audit.to_csv(audit_path, index=False)
horizon_batches.to_csv(batch_path, index=False)

print(submission_path)
print(audit_path)
print(batch_path)
print('Horizon safety audit rows:', len(horizon_audit))
print('History-after-decision leaks:', int(horizon_audit['history_after_effective_decision'].sum()))
print('Prediction-window violations:', int(horizon_audit['forecast_outside_prediction_window'].sum()))
print('Dynamic-covariate-after-context leaks:', int(horizon_audit['dynamic_covariate_after_context'].sum()))
print('Rows by route:')
display(strict_submission.groupby(['model_route', 'horizon'], observed=True).size().rename('rows').reset_index())
final_submission.head()


Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


cutoff=2024-10-02 rows= 140 items=140 context_max=2024-10-02 max_offset=30


Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


cutoff=2024-10-03 rows= 140 items=140 context_max=2024-10-03 max_offset=30


Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


...

Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


cutoff=2024-10-30 rows= 280 items=280 context_max=2024-10-30 max_offset=30


Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


cutoff=2024-10-31 rows=20720 items=420 context_max=2024-10-31 max_offset=61
/Users/moi/PROJECT/Expresso/outputs/autogluon_weighted_leakage_safe_horizon/submission_weighted_strict_horizon_safe_no_target_lift.csv
/Users/moi/PROJECT/Expresso/outputs/autogluon_weighted_leakage_safe_horizon/horizon_safe_prediction_audit_no_target_lift.csv
/Users/moi/PROJECT/Expresso/outputs/autogluon_weighted_leakage_safe_horizon/horizon_safe_prediction_batches_no_target_lift.csv
Horizon safety audit rows: 25620
History-after-decision leaks: 0
Prediction-window violations: 0
Dynamic-covariate-after-context leaks: 0


,id,units_sold_predicted
14224,10_Bakery_2024-11-01_1d,56.941054
63,10_Bakery_2024-11-01_1m,54.073896
3346,10_Bakery_2024-11-01_7d,54.228857
14225,10_Bakery_2024-11-02_1d,77.013528
203,10_Bakery_2024-11-02_1m,75.015635


## 8. Local Backtest / Sanity Checks

This section does lightweight checks on the forecast distribution versus recent history. For a full backtest, rerun the same pipeline with `TRAIN_END` moved to 2024-08-31 and prediction length 61 days.

In [22]:
recent_unique = train_df.drop_duplicates(['store_id', 'category', 'timestamp'])
recent = recent_unique[recent_unique['timestamp'].ge(TRAIN_END - pd.Timedelta(days=60))]
recent_summary = recent.groupby('category', observed=True)['units_sold'].agg(['mean', 'median', 'sum']).reset_index()
forecast_by_category = strict_submission.groupby('category', observed=True)['units_sold_predicted'].agg(['mean', 'median', 'sum']).reset_index()
check = recent_summary.merge(forecast_by_category, on='category', suffixes=('_recent_61d', '_forecast_rows'))
check['mean_ratio_forecast_to_recent'] = check['mean_forecast_rows'] / check['mean_recent_61d']
check.to_csv(OUTPUT_DIR / 'forecast_category_sanity_check.csv', index=False)

horizon_variation = (
    strict_submission.pivot_table(index=['store_id', 'category', 'forecast_date'], columns='horizon', values='units_sold_predicted', aggfunc='first')
    .reset_index()
)
horizon_variation['horizon_pred_range'] = horizon_variation[['1d', '7d', '1m']].max(axis=1) - horizon_variation[['1d', '7d', '1m']].min(axis=1)
horizon_variation.to_csv(OUTPUT_DIR / 'horizon_prediction_variation.csv', index=False)
print('Share of store/category/date rows with horizon-specific predictions:', horizon_variation['horizon_pred_range'].gt(1e-8).mean())
check


Share of store/category/date rows with horizon-specific predictions: 1.0


,category,mean_recent_61d,median_recent_61d,sum_recent_61d,mean_forecast_rows,median_forecast_rows,sum_forecast_rows,mean_ratio_forecast_to_recent
0,Coffee,143.717213,129.5,175335.0,165.420171,147.916407,605437.827556,1.151012
1,Tea,44.838525,39.0,54703.0,51.551722,45.300644,188679.303164,1.149719
2,Bakery,35.202459,31.0,42947.0,40.412276,36.865519,147908.929552,1.147996
3,Savory Bakery,28.209016,25.0,34415.0,32.236022,29.420725,117983.841916,1.142756
4,Chocolate & Milk,20.626230,18.0,25164.0,23.538533,21.122209,86151.029935,1.141194
5,Juice & Smoothie,12.401639,10.5,15130.0,14.102113,12.444272,51613.733172,1.137117
6,Merchandise,7.340984,6.0,8956.0,8.233972,7.479231,30136.337669,1.121644


## 9. Feature Importance

AutoGluon computes permutation importance by measuring the WAPE degradation after perturbing each feature. This can take longer than fitting, so `subsample_size` is kept modest by default.

In [27]:
if RUN_FEATURE_IMPORTANCE:
    for route, predictor in PREDICTORS_BY_ROUTE.items():
        artifacts = ARTIFACTS_BY_ROUTE[route]
        feature_importance = predictor.feature_importance(
            data=artifacts['train_tsdf'],
            method='permutation',
            subsample_size=50,
            num_iterations=1,
            time_limit=FEATURE_IMPORTANCE_TIME_LIMIT_SECONDS,
        )
        feature_importance_path = OUTPUT_DIR / f'feature_importance_{route}.csv'
        feature_importance.to_csv(feature_importance_path)
        print(feature_importance_path)
        display(feature_importance.head(30))
else:
    for route in ['short', 'long']:
        feature_importance_path = OUTPUT_DIR / f'feature_importance_{route}.csv'
        if feature_importance_path.exists():
            feature_importance = pd.read_csv(feature_importance_path, index_col=0)
            print(feature_importance_path)
            display(feature_importance.head(30))
        else:
            print(f'Feature importance file not found: {feature_importance_path}; set RUN_FEATURE_IMPORTANCE=True to compute it.')


Computing feature importance


/Users/moi/PROJECT/Expresso/outputs/autogluon_weighted_leakage_safe_horizon/feature_importance.csv


,importance,stdev,n,p99_low,p99_high
is_rainy_season,0.838818,NaN,1.0,NaN,NaN
is_holiday,0.758622,NaN,1.0,NaN,NaN
day_of_week,0.730225,NaN,1.0,NaN,NaN
kaggle_event_store_relevance_max,0.527653,NaN,1.0,NaN,NaN
is_payday,0.325187,NaN,1.0,NaN,NaN
max_discount_pct,0.272400,NaN,1.0,NaN,NaN
sin_doy,0.260062,NaN,1.0,NaN,NaN
store_id,0.227435,NaN,1.0,NaN,NaN
dow_num,0.219987,NaN,1.0,NaN,NaN
promo_type_count,0.198861,NaN,1.0,NaN,NaN


## 10. Save Source Reference Manifest

This keeps the external-data references alongside model artifacts.

In [28]:
reference_markdown = '''# AutoGluon External Feature References

Generated by `Expresso-AutoGluon-External-Features.ipynb`.

| Feature | Source | URL | Modeling use |
|---|---|---|---|
| Bangkok weather archive | Open-Meteo Historical Weather API | https://open-meteo.com/en/docs/historical-weather-api | Daily temperature, humidity, precipitation. |
| Thailand observed climatology | World Bank Climate Change Knowledge Portal | https://climateknowledgeportal.worldbank.org/country/thailand/climate-data-historical | Leakage-safe monthly weather fallback and climate context. |
| PTT/OR oil prices | PTTOR OilPrice SOAP service | https://orapiweb.pttor.com/oilservice/OilPrice.asmx?op=GetOilPrice | Daily PTT gasoline/diesel prices for PTT ecosystem store traffic and household fuel pressure. |
| Competitor oil prices | Bangchak historical retail oil prices | https://www.bangchak.co.th/en/oilprice/historical | Competitor price deltas versus PTT. |
| Thai holidays | Bank of Thailand financial institution holidays | https://www.bot.or.th/en/financial-institutions-holiday.html | Holiday and long-weekend calendar features. |
| Civil servant / pension payroll | Public payroll calendars / Comptroller General summaries | https://en.moneyandbanking.co.th/2024/141179/ | Paydate, pre-payday, and post-payday spending features. |
| Bangkok events | Visit Bangkok festival calendar | https://visit.bangkok.go.th/festival-calendar | Motor Expo, Red Cross Fair, and other citywide event shocks. |
| Existing project event layer | `analysis/coffee_hackathon_special_events.py` | local file | Store-type/category event relevance scores. |
| Inthanin Coffee context | Bangchak / Inthanin public reports | https://www.inthanincoffee.com/ and Bangchak investor materials | Branch-count and Bangchak-linked station coffee competitor pressure proxy. |
| PunThai Coffee context | PTG Energy one report / investor materials | https://www.pt.co.th/ and PTG investor materials | Branch-count, gas-station/outside-station mix, and PTG-linked coffee competitor pressure proxy. |
| PT Max / PTG oil context | PTG / PT Max Station public oil-price pages when available | https://www.pt.co.th/ | Potential future extension; notebook uses cacheable PTT/Bangchak historical prices for 2023-2024. |
'''
ref_path = OUTPUT_DIR / 'autogluon_external_feature_references.md'
ref_path.write_text(reference_markdown, encoding='utf-8')
print(ref_path)

/Users/moi/PROJECT/Expresso/outputs/autogluon_weighted_leakage_safe_horizon/autogluon_external_feature_references.md
